<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG_Cell_7B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES-RAG Experiment 2 — Cell 7B1


In [3]:
from collections import OrderedDict
from pathlib import Path
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# CELL 7B1 — SCORE-BLIND EVIDENCE / QUESTION PREFLIGHT
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG_Cell_7B1_V3.ipynb"
CELL_ID = "7B1"
PACKAGE_VERSION = "v1"

CONFIG_DIR = ROOT / "configs" / "stage7_rag"
TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in (CONFIG_DIR, TABLE_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# FROZEN CELL 7B0 INPUTS
# ============================================================

CELL_7B0_PROTOCOL = (
    CONFIG_DIR
    / "cell_7b0_downstream_rag_protocol_v1.json"
)

CELL_7B0_CONDITIONS = (
    TABLE_DIR
    / "cell_7b0_experimental_condition_inventory_v1.csv"
)

CELL_7B0_METRICS = (
    TABLE_DIR
    / "cell_7b0_evaluation_metric_inventory_v1.csv"
)

CELL_7B0_CONTROLS = (
    TABLE_DIR
    / "cell_7b0_leakage_and_invariance_control_inventory_v1.csv"
)

CELL_7B0_QC = (
    QC_DIR
    / "cell_7b0_downstream_rag_protocol_qc_v1.json"
)

CELL_7B0_MANIFEST = (
    CONFIG_DIR
    / "cell_7b0_downstream_rag_protocol_manifest_v1.json"
)


# ============================================================
# FROZEN RAW-T1 ANCESTRY
# ============================================================

CELL_7A1_MANIFEST = (
    CONFIG_DIR
    / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)

T1_PARQUET = (
    ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

T1_FREEZE_MANIFEST = (
    ROOT
    / "configs"
    / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)


# This path is defined only as a prohibition guard.
# It is never opened, hashed, inspected, or loaded in Cell 7B1.

PROHIBITED_CELL_7A3_SCORE_TABLE = (
    ROOT
    / "data_processed"
    / "stage7_rag"
    / "cell_7a3_t1_frozen_ges_and_metadata_scores_v1.parquet"
)


EXPECTED_HASHES = OrderedDict(
    [
        (
            "cell_7b0_protocol",
            "db4fe2b527e37aba4b4ea5967517c3896e933f989a7ad264406077c4298bd849",
        ),
        (
            "cell_7b0_conditions",
            "ca1b51cbf21e9401da90c9fa0688e71f9d3755cbd467220b9409c2b79425d03c",
        ),
        (
            "cell_7b0_metrics",
            "e4e3a00870cb8e1430f01a3635ee7b6a77b7ceb3bc2ca0fad495e170cf451223",
        ),
        (
            "cell_7b0_controls",
            "bc8e8ec7f1d120bc57589ae544d87b0ab7f8fcc26755cf89c34e782787bcf049",
        ),
        (
            "cell_7b0_qc",
            "126207c1552ba4a65eb4a60e36aef1e95500228b4bfc38b77f8d773dd3a3d776",
        ),
        (
            "cell_7b0_manifest",
            "df9342b8a2fb641f4cb68ff18ae9568bb7f63eff3fcc5e1ae1106601fa59e42a",
        ),
        (
            "cell_7a1_manifest",
            "84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d",
        ),
        (
            "t1_parquet",
            "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c",
        ),
        (
            "t1_freeze_manifest",
            "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e",
        ),
    ]
)


INPUT_PATHS = OrderedDict(
    [
        ("cell_7b0_protocol", CELL_7B0_PROTOCOL),
        ("cell_7b0_conditions", CELL_7B0_CONDITIONS),
        ("cell_7b0_metrics", CELL_7B0_METRICS),
        ("cell_7b0_controls", CELL_7B0_CONTROLS),
        ("cell_7b0_qc", CELL_7B0_QC),
        ("cell_7b0_manifest", CELL_7B0_MANIFEST),
        ("cell_7a1_manifest", CELL_7A1_MANIFEST),
        ("t1_parquet", T1_PARQUET),
        ("t1_freeze_manifest", T1_FREEZE_MANIFEST),
    ]
)


EXPECTED_CELL_7B0_DECISION = (
    "PASS_STAGE7B0_DOWNSTREAM_RAG_PROTOCOL_FROZEN_CHECKSUM_PROTECTED_"
    "STAGE7A3_REVERIFIED_PRIMARY_SOFT_RERANKING_AND_MANDATORY_COMPARATOR_"
    "ARMS_PRESPECIFIED_NO_EVIDENCE_PACKETS_CORPUS_EMBEDDINGS_RETRIEVAL_"
    "QUESTIONS_PROMPTS_OR_LLM_CELL7B1_SCORE_BLIND_EVIDENCE_AND_QUESTION_"
    "PREFLIGHT_ONLY_AUTHORIZED"
)


EXPECTED_CELL_7A1_DECISION = (
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)


EXPECTED_ROWS = 100_920
EXPECTED_COLUMNS = 36
EXPECTED_NESTED_SCVS = 145_400
EXPECTED_CONFLICT_POSITIVE = 6_602
EXPECTED_EMPTY_CONDITION_IDS = 730


EXPECTED_GENE_COUNTS = OrderedDict(
    [
        ("BRCA1", 32_603),
        ("BRCA2", 49_221),
        ("MLH1", 13_684),
        ("EGFR", 5_412),
    ]
)


EXPECTED_AXIS_COUNTS = OrderedDict(
    [
        ("GermlineClassification", 97_526),
        ("OncogenicityClassification", 52),
        ("SomaticClinicalImpact", 25),
        ("NoClassification", 3_317),
    ]
)


# ============================================================
# OUTPUTS
# ============================================================

OUTPUTS = OrderedDict(
    [
        (
            "source_inventory",
            TABLE_DIR
            / "cell_7b1_score_blind_source_inventory_v1.csv",
        ),
        (
            "field_derivability",
            TABLE_DIR
            / "cell_7b1_evidence_field_derivability_inventory_v1.csv",
        ),
        (
            "gene_axis_inventory",
            TABLE_DIR
            / "cell_7b1_gene_axis_eligibility_inventory_v1.csv",
        ),
        (
            "question_strata",
            TABLE_DIR
            / "cell_7b1_question_stratum_availability_inventory_v1.csv",
        ),
        (
            "preflight_report",
            QC_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_report_v1.json",
        ),
        (
            "qc",
            QC_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_qc_v1.json",
        ),
        (
            "manifest",
            CONFIG_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_manifest_v1.json",
        ),
    ]
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def sidecar_path(path):
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    values = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not values:
        raise ValueError(
            f"No SHA-256 found in sidecar: {path}"
        )

    return values[0].lower()


def sidecar_is_valid(path):
    path = Path(path)
    checksum_sidecar = sidecar_path(path)

    return (
        path.exists()
        and checksum_sidecar.exists()
        and read_sidecar_hash(checksum_sidecar)
        == sha256_file(path)
    )


def verify_exact_hash(
    label,
    path,
    expected,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing frozen artifact for {label}: {path}"
        )

    observed = sha256_file(path)

    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}\n"
            f"Expected: {expected}\n"
            f"Observed: {observed}\n"
            f"Path: {path}"
        )

    return observed


def json_native(value):
    if isinstance(value, dict):
        return {
            str(key): json_native(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_native(item)
            for item in value
        ]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None

        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if value is pd.NA:
        return None

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def stable_write_bytes(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp-"
        f"{os.getpid()}-"
        f"{time.time_ns()}"
    )

    temporary_path.write_bytes(payload)

    proposed_hash = sha256_file(
        temporary_path
    )

    if path.exists():
        existing_hash = sha256_file(path)

        if existing_hash != proposed_hash:
            temporary_path.unlink(
                missing_ok=True
            )

            raise RuntimeError(
                "Refusing to overwrite a nonidentical "
                "frozen Cell 7B1 artifact.\n"
                f"Path: {path}\n"
                f"Existing: {existing_hash}\n"
                f"Proposed: {proposed_hash}"
            )

        temporary_path.unlink(
            missing_ok=True
        )

    else:
        os.replace(
            temporary_path,
            path,
        )

    return sha256_file(path)


def stable_write_json(
    path,
    payload,
):
    encoded = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        encoded,
    )


def stable_write_csv(
    path,
    dataframe,
):
    encoded = dataframe.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        encoded,
    )


def write_sidecar(path):
    payload = (
        f"{sha256_file(path)}  "
        f"{Path(path).name}\n"
    ).encode("utf-8")

    return stable_write_bytes(
        sidecar_path(path),
        payload,
    )


def normalize_qc_count(
    value,
    field_name,
):
    if isinstance(value, bool):
        return int(value)

    if isinstance(
        value,
        (int, float),
    ):
        return int(value)

    if isinstance(
        value,
        (list, tuple, set, dict),
    ):
        return len(value)

    if (
        isinstance(value, str)
        and value.strip().isdigit()
    ):
        return int(value.strip())

    if value is None:
        return 0

    raise TypeError(
        f"Unsupported QC field type for "
        f"{field_name}: "
        f"{type(value).__name__}"
    )


def qc_summary_counts(payload):
    passed = normalize_qc_count(
        payload.get(
            "passed_checks",
            0,
        ),
        "passed_checks",
    )

    failed = normalize_qc_count(
        payload.get(
            "failed_checks",
            0,
        ),
        "failed_checks",
    )

    total_raw = payload.get(
        "total_checks"
    )

    if total_raw is None:
        checks = payload.get(
            "checks"
        )

        if isinstance(
            checks,
            (list, tuple, dict),
        ):
            total = len(checks)

        else:
            total = passed + failed

    else:
        total = normalize_qc_count(
            total_raw,
            "total_checks",
        )

    return (
        passed,
        failed,
        total,
    )


def normalize_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


def resolve_column(
    columns,
    aliases,
    required=False,
    token_fallback=None,
):
    normalized = {
        normalize_column_name(column): str(column)
        for column in columns
    }

    for alias in aliases:
        normalized_alias = normalize_column_name(
            alias
        )

        if normalized_alias in normalized:
            return normalized[
                normalized_alias
            ]

    if token_fallback:
        matches = [
            str(column)
            for column in columns
            if all(
                token
                in normalize_column_name(column)
                for token in token_fallback
            )
        ]

        if len(matches) == 1:
            return matches[0]

    if required:
        raise KeyError(
            "Could not resolve required column. "
            f"Aliases={aliases}; "
            f"tokens={token_fallback}; "
            f"available={list(columns)}"
        )

    return None


def parse_json_value(value):
    if (
        value is None
        or value is pd.NA
    ):
        return (
            None,
            "missing",
        )

    try:
        if pd.isna(value):
            return (
                None,
                "missing",
            )

    except Exception:
        pass

    if isinstance(
        value,
        (dict, list),
    ):
        return (
            value,
            "native",
        )

    text = str(value).strip()

    if not text:
        return (
            None,
            "blank",
        )

    try:
        return (
            json.loads(text),
            "parsed",
        )

    except json.JSONDecodeError:
        return (
            None,
            "parse_error",
        )


def normalize_gene(value):
    parsed, status = parse_json_value(
        value
    )

    if status == "parse_error":
        parsed = [
            str(value).strip()
        ]

    if isinstance(parsed, str):
        parsed = [parsed]

    if not isinstance(parsed, list):
        return ""

    allowed = set(
        EXPECTED_GENE_COUNTS
    )

    genes = sorted(
        {
            str(item).strip().upper()
            for item in parsed
            if str(item).strip().upper()
            in allowed
        }
    )

    if len(genes) == 1:
        return genes[0]

    return ""


def boolish(value):
    if (
        value is None
        or value is pd.NA
    ):
        return np.nan

    try:
        if pd.isna(value):
            return np.nan

    except Exception:
        pass

    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return bool(value)

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):
        if float(value) == 1.0:
            return True

        if float(value) == 0.0:
            return False

        return np.nan

    text = str(value).strip().lower()

    if text in {
        "true",
        "t",
        "yes",
        "y",
        "1",
    }:
        return True

    if text in {
        "false",
        "f",
        "no",
        "n",
        "0",
    }:
        return False

    return np.nan


def value_is_present(value):
    if (
        value is None
        or value is pd.NA
    ):
        return False

    try:
        if pd.isna(value):
            return False

    except Exception:
        pass

    if isinstance(value, str):
        return bool(
            value.strip()
        )

    if isinstance(
        value,
        (
            list,
            dict,
            tuple,
            set,
        ),
    ):
        return len(value) > 0

    return True


def json_list_length(value):
    parsed, status = parse_json_value(
        value
    )

    if status == "parse_error":
        return (
            np.nan,
            status,
        )

    if parsed is None:
        return (
            0,
            status,
        )

    if isinstance(parsed, list):
        return (
            len(parsed),
            status,
        )

    return (
        np.nan,
        "not_list",
    )


def json_dict_positive_key_count(value):
    parsed, status = parse_json_value(
        value
    )

    if status == "parse_error":
        return (
            np.nan,
            status,
        )

    if parsed is None:
        return (
            0,
            status,
        )

    if not isinstance(parsed, dict):
        return (
            np.nan,
            "not_dict",
        )

    positive = 0

    for item in parsed.values():
        try:
            number = float(item)

        except (
            TypeError,
            ValueError,
        ):
            return (
                np.nan,
                "invalid_numeric",
            )

        if (
            not np.isfinite(number)
            or number < 0
        ):
            return (
                np.nan,
                "invalid_numeric",
            )

        positive += int(
            number > 0
        )

    return (
        positive,
        status,
    )


# ============================================================
# VERIFY CELL 7B0 AND RAW-T1 AUTHORIZATION
# ============================================================

observed_hashes = OrderedDict()

for key, path in INPUT_PATHS.items():
    observed_hashes[key] = verify_exact_hash(
        key,
        path,
        EXPECTED_HASHES[key],
    )

    if not sidecar_is_valid(path):
        raise AssertionError(
            "Missing or invalid SHA-256 sidecar "
            f"for {key}: {path}"
        )


cell_7b0_manifest = json.loads(
    CELL_7B0_MANIFEST.read_text(
        encoding="utf-8"
    )
)

cell_7b0_qc = json.loads(
    CELL_7B0_QC.read_text(
        encoding="utf-8"
    )
)

cell_7a1_manifest = json.loads(
    CELL_7A1_MANIFEST.read_text(
        encoding="utf-8"
    )
)


(
    manifest_qc_passed,
    manifest_qc_failed,
    manifest_qc_total,
) = qc_summary_counts(
    cell_7b0_manifest.get(
        "qc",
        {},
    )
)


(
    record_qc_passed,
    record_qc_failed,
    record_qc_total,
) = qc_summary_counts(
    cell_7b0_qc
)


if (
    cell_7b0_manifest.get(
        "terminal_decision"
    )
    != EXPECTED_CELL_7B0_DECISION
):
    raise AssertionError(
        "Cell 7B0 terminal decision is not "
        "the expected frozen PASS."
    )


next_7b0 = cell_7b0_manifest.get(
    "next_authorized_cell",
    {},
)


if (
    next_7b0.get("cell_id")
    != "7B1"
):
    raise AssertionError(
        "Cell 7B0 does not authorize Cell 7B1."
    )


if (
    next_7b0.get(
        "may_load_cell_7a3_scores"
    )
    is not False
):
    raise AssertionError(
        "Cell 7B0 score-blind restriction "
        "is not preserved."
    )


if (
    cell_7a1_manifest.get(
        "terminal_decision"
    )
    != EXPECTED_CELL_7A1_DECISION
):
    raise AssertionError(
        "Cell 7A1 source-preflight ancestry "
        "is not the expected PASS."
    )


immutable_hashes_before = {
    key: sha256_file(path)
    for key, path in INPUT_PATHS.items()
}


# ============================================================
# LOAD ONLY RAW T1
# ============================================================

parquet_metadata = pq.ParquetFile(
    T1_PARQUET
).metadata


if (
    int(parquet_metadata.num_rows)
    != EXPECTED_ROWS
):
    raise AssertionError(
        f"T1 rows={parquet_metadata.num_rows}; "
        f"expected {EXPECTED_ROWS}."
    )


if (
    int(parquet_metadata.num_columns)
    != EXPECTED_COLUMNS
):
    raise AssertionError(
        f"T1 columns={parquet_metadata.num_columns}; "
        f"expected {EXPECTED_COLUMNS}."
    )


t1 = pd.read_parquet(
    T1_PARQUET
).copy()


resolved = OrderedDict(
    [
        (
            "rcv_accession",
            resolve_column(
                t1.columns,
                [
                    "rcv_accession",
                    "t1_rcv_accession",
                ],
                required=True,
                token_fallback=[
                    "rcv",
                    "accession",
                ],
            ),
        ),
        (
            "vcv_accession",
            resolve_column(
                t1.columns,
                [
                    "vcv_accession",
                    "variation_archive_accession",
                ],
                token_fallback=[
                    "vcv",
                    "accession",
                ],
            ),
        ),
        (
            "variation_id",
            resolve_column(
                t1.columns,
                [
                    "variation_id",
                    "variationid",
                ],
                token_fallback=[
                    "variation",
                    "id",
                ],
            ),
        ),
        (
            "variation_name",
            resolve_column(
                t1.columns,
                [
                    "variation_name",
                    "variant_name",
                    "name",
                ],
                token_fallback=[
                    "variation",
                    "name",
                ],
            ),
        ),
        (
            "target_gene",
            resolve_column(
                t1.columns,
                [
                    "target_genes_json",
                    "target_gene",
                    "gene",
                ],
                required=True,
            ),
        ),
        (
            "classification_axis",
            resolve_column(
                t1.columns,
                [
                    "aggregate_classification_axis",
                    "classification_axis",
                ],
                required=True,
            ),
        ),
        (
            "condition_names",
            resolve_column(
                t1.columns,
                [
                    "condition_names_json",
                    "condition_names",
                    "trait_names_json",
                ],
                token_fallback=[
                    "condition",
                    "name",
                ],
            ),
        ),
        (
            "condition_ids",
            resolve_column(
                t1.columns,
                [
                    "condition_ids_json",
                    "condition_identifiers_json",
                    "condition_xrefs_json",
                    "trait_ids_json",
                ],
                token_fallback=[
                    "condition",
                    "id",
                ],
            ),
        ),
        (
            "aggregate_classification",
            resolve_column(
                t1.columns,
                [
                    "aggregate_classification",
                    "aggregate_clinical_significance",
                    "aggregate_classification_description",
                    "clinical_significance",
                ],
                token_fallback=[
                    "aggregate",
                    "classification",
                ],
            ),
        ),
        (
            "aggregate_classification_group",
            resolve_column(
                t1.columns,
                [
                    "aggregate_classification_group",
                    "classification_group",
                    "aggregate_group",
                ],
                token_fallback=[
                    "classification",
                    "group",
                ],
            ),
        ),
        (
            "aggregate_review_status",
            resolve_column(
                t1.columns,
                [
                    "aggregate_review_status",
                    "review_status",
                ],
                token_fallback=[
                    "review",
                    "status",
                ],
            ),
        ),
        (
            "aggregate_review_stars",
            resolve_column(
                t1.columns,
                [
                    "aggregate_review_stars",
                    "review_stars",
                ],
                required=True,
                token_fallback=[
                    "review",
                    "star",
                ],
            ),
        ),
        (
            "aggregate_conflict_flag",
            resolve_column(
                t1.columns,
                [
                    "aggregate_conflict_flag",
                    "conflict_flag",
                ],
                required=True,
                token_fallback=[
                    "conflict",
                    "flag",
                ],
            ),
        ),
        (
            "aggregate_last_evaluated",
            resolve_column(
                t1.columns,
                [
                    "aggregate_last_evaluated",
                    "last_evaluated",
                ],
                required=True,
                token_fallback=[
                    "last",
                    "evaluated",
                ],
            ),
        ),
        (
            "scv_count",
            resolve_column(
                t1.columns,
                [
                    "scv_count_xml",
                    "scv_count",
                ],
                required=True,
                token_fallback=[
                    "scv",
                    "count",
                ],
            ),
        ),
        (
            "unique_submitter_count",
            resolve_column(
                t1.columns,
                [
                    "unique_submitter_count_xml",
                    "unique_submitter_count",
                ],
                required=True,
                token_fallback=[
                    "submitter",
                    "count",
                ],
            ),
        ),
        (
            "submitter_ids",
            resolve_column(
                t1.columns,
                [
                    "submitter_ids_json",
                    "submitter_org_ids_json",
                    "unique_submitter_ids_json",
                ],
                token_fallback=[
                    "submitter",
                    "id",
                ],
            ),
        ),
        (
            "scv_group_counts",
            resolve_column(
                t1.columns,
                [
                    "scv_group_counts_json",
                    "group_counts_json",
                ],
                required=True,
                token_fallback=[
                    "scv",
                    "group",
                    "count",
                ],
            ),
        ),
        (
            "nested_scvs",
            resolve_column(
                t1.columns,
                [
                    "scv_records_json",
                    "nested_scvs_json",
                    "nested_scv_assertions_json",
                    "scvs_json",
                    "nested_scvs",
                ],
                required=True,
                token_fallback=[
                    "scv",
                    "records",
                ],
            ),
        ),
        (
            "embedded_cutoff_date",
            resolve_column(
                t1.columns,
                [
                    "embedded_data_cutoff_date",
                    "data_cutoff_date",
                    "embedded_cutoff_date",
                ],
                required=True,
                token_fallback=[
                    "cutoff",
                    "date",
                ],
            ),
        ),
        (
            "release_label",
            resolve_column(
                t1.columns,
                [
                    "archive_release_label",
                    "release_label",
                    "clinvar_release_label",
                ],
                token_fallback=[
                    "release",
                    "label",
                ],
            ),
        ),
        (
            "source_filename",
            resolve_column(
                t1.columns,
                [
                    "source_filename",
                    "source_file_name",
                    "xml_source_filename",
                ],
                token_fallback=[
                    "source",
                    "filename",
                ],
            ),
        ),
        (
            "source_sha256",
            resolve_column(
                t1.columns,
                [
                    "source_sha256",
                    "source_file_sha256",
                    "xml_source_sha256",
                ],
                token_fallback=[
                    "source",
                    "sha256",
                ],
            ),
        ),
    ]
)


if (
    resolved["aggregate_classification"]
    is None
    and resolved[
        "aggregate_classification_group"
    ]
    is None
):
    raise KeyError(
        "Aggregate classification fields "
        "could not be resolved."
    )


if (
    resolved["condition_names"]
    is None
    and resolved["condition_ids"]
    is None
):
    raise KeyError(
        "Condition fields could not be resolved."
    )


# ============================================================
# SCORE-BLIND DERIVATIONS
# ============================================================

rcv = (
    t1[
        resolved["rcv_accession"]
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)


gene = t1[
    resolved["target_gene"]
].map(normalize_gene)


axis = (
    t1[
        resolved["classification_axis"]
    ]
    .astype("string")
    .fillna("NoClassification")
    .str.strip()
    .replace(
        "",
        "NoClassification",
    )
)


conflict = t1[
    resolved["aggregate_conflict_flag"]
].map(boolish)


review_stars = pd.to_numeric(
    t1[
        resolved["aggregate_review_stars"]
    ],
    errors="coerce",
)


last_evaluated = pd.to_datetime(
    t1[
        resolved[
            "aggregate_last_evaluated"
        ]
    ],
    errors="coerce",
    utc=True,
)


scv_count = pd.to_numeric(
    t1[
        resolved["scv_count"]
    ],
    errors="coerce",
)


submitter_count = pd.to_numeric(
    t1[
        resolved["unique_submitter_count"]
    ],
    errors="coerce",
)


nested_results = t1[
    resolved["nested_scvs"]
].map(json_list_length)


nested_scv_count = pd.Series(
    [
        item[0]
        for item in nested_results
    ],
    index=t1.index,
    dtype=float,
)


nested_status = pd.Series(
    [
        item[1]
        for item in nested_results
    ],
    index=t1.index,
    dtype="string",
)


group_results = t1[
    resolved["scv_group_counts"]
].map(json_dict_positive_key_count)


positive_group_count = pd.Series(
    [
        item[0]
        for item in group_results
    ],
    index=t1.index,
    dtype=float,
)


group_status = pd.Series(
    [
        item[1]
        for item in group_results
    ],
    index=t1.index,
    dtype="string",
)


if resolved["condition_ids"] is not None:
    condition_id_results = t1[
        resolved["condition_ids"]
    ].map(json_list_length)

    condition_id_count = pd.Series(
        [
            item[0]
            for item in condition_id_results
        ],
        index=t1.index,
        dtype=float,
    )

else:
    condition_id_count = pd.Series(
        np.nan,
        index=t1.index,
        dtype=float,
    )


if resolved["condition_names"] is not None:
    condition_name_present = t1[
        resolved["condition_names"]
    ].map(value_is_present)

else:
    condition_name_present = pd.Series(
        False,
        index=t1.index,
    )


if (
    resolved["aggregate_classification"]
    is not None
):
    classification_present = t1[
        resolved["aggregate_classification"]
    ].map(value_is_present)

else:
    classification_present = pd.Series(
        False,
        index=t1.index,
    )


if (
    resolved[
        "aggregate_classification_group"
    ]
    is not None
):
    classification_group_present = t1[
        resolved[
            "aggregate_classification_group"
        ]
    ].map(value_is_present)

else:
    classification_group_present = pd.Series(
        False,
        index=t1.index,
    )


condition_present = (
    condition_name_present
    | condition_id_count
    .fillna(0)
    .gt(0)
)


interpretation_present = (
    classification_present
    | classification_group_present
)


nested_available = (
    nested_scv_count
    .fillna(0)
    .gt(0)
)


summary_eligible = (
    rcv.notna()
    & rcv.ne("")
    & gene.ne("")
    & condition_present
    & interpretation_present
    & nested_available
)


conflict_eligible = (
    summary_eligible
    & (
        conflict
        .fillna(False)
        .astype(bool)
        | positive_group_count
        .fillna(0)
        .gt(1)
    )
)


rigor_eligible = (
    summary_eligible
    & review_stars.notna()
    & review_stars.between(
        0,
        4,
        inclusive="both",
    )
    & submitter_count
    .fillna(0)
    .ge(1)
    & scv_count
    .fillna(0)
    .ge(1)
)


uncertainty_eligible = (
    summary_eligible
    & (
        axis.eq(
            "NoClassification"
        )
        | conflict
        .fillna(False)
        .astype(bool)
        | condition_id_count
        .fillna(0)
        .eq(0)
        | last_evaluated.isna()
        | review_stars
        .fillna(0)
        .le(1)
        | positive_group_count
        .fillna(0)
        .gt(1)
    )
)


eligibility_masks = OrderedDict(
    [
        (
            "aggregate_interpretation_summary",
            summary_eligible,
        ),
        (
            "conflict_recognition",
            conflict_eligible,
        ),
        (
            "evidence_rigor_and_provenance",
            rigor_eligible,
        ),
        (
            "uncertainty_or_abstention",
            uncertainty_eligible,
        ),
    ]
)


# ============================================================
# SOURCE INVENTORY
# ============================================================

source_inventory_rows = []

for key, path in INPUT_PATHS.items():
    record = {
        "artifact_key": key,
        "path": str(path),
        "sha256": observed_hashes[key],
        "sidecar_verified": sidecar_is_valid(
            path
        ),
        "bytes": int(
            path.stat().st_size
        ),
        "rows": None,
        "columns": None,
        "loaded_in_cell_7b1": (
            key == "t1_parquet"
        ),
    }

    if path.suffix.lower() == ".parquet":
        metadata = pq.ParquetFile(
            path
        ).metadata

        record["rows"] = int(
            metadata.num_rows
        )

        record["columns"] = int(
            metadata.num_columns
        )

    source_inventory_rows.append(
        record
    )


source_inventory = pd.DataFrame(
    source_inventory_rows
)


# ============================================================
# FIELD DERIVABILITY INVENTORY
# ============================================================

field_specs = [
    (
        "rcv_accession",
        "identifier",
        "top_level",
        True,
        "direct",
    ),
    (
        "vcv_accession",
        "identifier",
        "top_level",
        False,
        "direct",
    ),
    (
        "variation_id",
        "identifier",
        "top_level",
        False,
        "direct",
    ),
    (
        "variation_name",
        "identifier",
        "top_level",
        False,
        "direct",
    ),
    (
        "target_gene",
        "identifier",
        "top_level",
        True,
        "direct",
    ),
    (
        "release_label",
        "provenance",
        "top_level_or_manifest",
        False,
        "direct_or_manifest",
    ),
    (
        "embedded_cutoff_date",
        "provenance",
        "top_level",
        True,
        "direct",
    ),
    (
        "source_filename",
        "provenance",
        "top_level_or_manifest",
        False,
        "direct_or_manifest",
    ),
    (
        "source_sha256",
        "provenance",
        "top_level_or_manifest",
        False,
        "direct_or_manifest",
    ),
    (
        "condition_names",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "condition_ids",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "aggregate_classification",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "aggregate_classification_group",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "classification_axis",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "aggregate_review_status",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "aggregate_review_stars",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "aggregate_conflict_flag",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "aggregate_last_evaluated",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "scv_count",
        "provenance",
        "top_level",
        True,
        "direct",
    ),
    (
        "unique_submitter_count",
        "provenance",
        "top_level",
        True,
        "direct",
    ),
    (
        "submitter_ids",
        "provenance",
        "top_level_or_nested",
        False,
        "direct_or_nested",
    ),
    (
        "scv_group_counts",
        "nested_summary",
        "top_level",
        True,
        "direct",
    ),
    (
        "nested_scv_assertions",
        "nested_evidence",
        "nested_scvs",
        True,
        "deterministically_parseable",
    ),
    (
        "nested_scv_review_statuses",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "nested_scv_classifications",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "nested_scv_last_evaluated",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "nested_scv_submitter_ids",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "evidence_packet_id",
        "derived_identifier",
        "later_formatting_policy",
        False,
        "derivable_later",
    ),
    (
        "semantic_evidence_text",
        "derived_text",
        "later_formatting_policy",
        False,
        "derivable_later",
    ),
    (
        "full_ges_p_stable_t1",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
    (
        "full_ges_instability_risk_t1",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
    (
        "no_star_ges_p_stable_t1",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
    (
        "combined_metadata_instability_risk",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
]


field_rows = []


for (
    field_name,
    category,
    source_class,
    required,
    derivation_status,
) in field_specs:

    resolved_column = resolved.get(
        field_name
    )

    if source_class == "nested_scvs":
        resolved_column = resolved[
            "nested_scvs"
        ]

        available = (
            resolved_column is not None
        )

        missing_rows = int(
            nested_scv_count
            .fillna(0)
            .eq(0)
            .sum()
        )

    elif source_class == "later_formatting_policy":
        available = True
        missing_rows = None

    elif source_class == "cell_7a3":
        available = True
        missing_rows = None
        resolved_column = None

    elif source_class == "top_level_or_manifest":
        available = (
            resolved_column is not None
            or T1_FREEZE_MANIFEST.exists()
        )

        if resolved_column is not None:
            missing_rows = int(
                (
                    ~t1[
                        resolved_column
                    ].map(
                        value_is_present
                    )
                ).sum()
            )

        else:
            missing_rows = None

    elif source_class == "top_level_or_nested":
        available = (
            resolved_column is not None
            or resolved["nested_scvs"]
            is not None
        )

        if resolved_column is not None:
            missing_rows = int(
                (
                    ~t1[
                        resolved_column
                    ].map(
                        value_is_present
                    )
                ).sum()
            )

        else:
            missing_rows = None

    else:
        available = (
            resolved_column is not None
        )

        if resolved_column is not None:
            missing_rows = int(
                (
                    ~t1[
                        resolved_column
                    ].map(
                        value_is_present
                    )
                ).sum()
            )

        else:
            missing_rows = None

    field_rows.append(
        {
            "field_name": field_name,
            "category": category,
            "source_class": source_class,
            "required_for_future_packet": bool(
                required
            ),
            "derivation_status": derivation_status,
            "resolved_source_column": resolved_column,
            "available_or_derivable": bool(
                available
            ),
            "missing_rows_if_directly_auditable": missing_rows,
            "loaded_or_materialized_in_cell_7b1": (
                source_class
                != "cell_7a3"
            ),
        }
    )


field_derivability = pd.DataFrame(
    field_rows
)


# ============================================================
# GENE × CLASSIFICATION-AXIS INVENTORY
# ============================================================

gene_axis_rows = []


for gene_name in EXPECTED_GENE_COUNTS:
    for axis_name in EXPECTED_AXIS_COUNTS:
        mask = (
            gene.eq(gene_name)
            & axis.eq(axis_name)
        )

        gene_axis_rows.append(
            {
                "target_gene": gene_name,
                "classification_axis": axis_name,
                "row_count": int(
                    mask.sum()
                ),
                "conflict_positive": int(
                    (
                        mask
                        & conflict
                        .fillna(False)
                        .astype(bool)
                    ).sum()
                ),
                "empty_structured_condition_ids": int(
                    (
                        mask
                        & condition_id_count
                        .fillna(0)
                        .eq(0)
                    ).sum()
                ),
                "nested_scv_count": int(
                    nested_scv_count.loc[
                        mask
                    ]
                    .fillna(0)
                    .sum()
                ),
            }
        )


gene_axis_inventory = pd.DataFrame(
    gene_axis_rows
)


# ============================================================
# QUESTION-STRATUM AVAILABILITY
# ============================================================

question_rows = []


for gene_name in EXPECTED_GENE_COUNTS:
    gene_mask = gene.eq(
        gene_name
    )

    for (
        question_type,
        eligibility_mask,
    ) in eligibility_masks.items():

        combined_mask = (
            gene_mask
            & eligibility_mask
        )

        axis_breakdown = {
            axis_name: int(
                (
                    combined_mask
                    & axis.eq(axis_name)
                ).sum()
            )
            for axis_name
            in EXPECTED_AXIS_COUNTS
        }

        question_rows.append(
            {
                "target_gene": gene_name,
                "question_type": question_type,
                "target_question_count": 5,
                "eligible_rcv_count": int(
                    combined_mask.sum()
                ),
                "feasible_for_target": bool(
                    combined_mask.sum()
                    >= 5
                ),
                "classification_axis_breakdown_json": json.dumps(
                    axis_breakdown,
                    sort_keys=True,
                    separators=(
                        ",",
                        ":",
                    ),
                ),
                "questions_selected_or_generated": False,
            }
        )


question_strata = pd.DataFrame(
    question_rows
)


# ============================================================
# FAIL-BEFORE-OUTPUT QC
# ============================================================

observed_gene_counts = (
    gene.value_counts(
        dropna=False
    ).to_dict()
)


observed_axis_counts = (
    axis.value_counts(
        dropna=False
    ).to_dict()
)


malformed_rcv = int(
    (
        ~rcv.str.fullmatch(
            r"RCV\d+(?:\.\d+)?",
            na=False,
        )
    ).sum()
)


duplicate_rcv = int(
    rcv.duplicated(
        keep=False
    ).sum()
)


nested_parse_errors = int(
    nested_status.eq(
        "parse_error"
    ).sum()
)


nested_non_lists = int(
    nested_status.eq(
        "not_list"
    ).sum()
)


group_parse_errors = int(
    group_status.eq(
        "parse_error"
    ).sum()
)


group_invalid = int(
    group_status.isin(
        [
            "not_dict",
            "invalid_numeric",
        ]
    ).sum()
)


nested_total = int(
    nested_scv_count
    .fillna(0)
    .sum()
)


scv_count_mismatches = int(
    (
        ~np.isclose(
            nested_scv_count.to_numpy(
                dtype=float
            ),
            scv_count.to_numpy(
                dtype=float
            ),
            rtol=0.0,
            atol=0.0,
            equal_nan=False,
        )
    ).sum()
)


conflict_positive = int(
    conflict
    .fillna(False)
    .astype(bool)
    .sum()
)


empty_condition_ids = int(
    condition_id_count
    .fillna(0)
    .eq(0)
    .sum()
)


all_required_fields_available = bool(
    field_derivability.loc[
        field_derivability[
            "required_for_future_packet"
        ],
        "available_or_derivable",
    ].all()
)


all_strata_feasible = bool(
    question_strata[
        "feasible_for_target"
    ].all()
)


prewrite_checks = OrderedDict(
    [
        (
            "cell_7b0_protocol_exact_hash",
            observed_hashes[
                "cell_7b0_protocol"
            ]
            == EXPECTED_HASHES[
                "cell_7b0_protocol"
            ],
        ),
        (
            "cell_7b0_manifest_exact_hash",
            observed_hashes[
                "cell_7b0_manifest"
            ]
            == EXPECTED_HASHES[
                "cell_7b0_manifest"
            ],
        ),
        (
            "cell_7b0_terminal_decision_exact",
            cell_7b0_manifest.get(
                "terminal_decision"
            )
            == EXPECTED_CELL_7B0_DECISION,
        ),
        (
            "cell_7b0_authorizes_7b1",
            next_7b0.get(
                "cell_id"
            )
            == "7B1",
        ),
        (
            "cell_7b0_manifest_qc_50_of_50",
            manifest_qc_passed == 50
            and manifest_qc_failed == 0
            and manifest_qc_total == 50,
        ),
        (
            "cell_7b0_record_qc_50_of_50",
            record_qc_passed == 50
            and record_qc_failed == 0
            and record_qc_total == 50,
        ),
        (
            "cell_7a1_manifest_exact_hash",
            observed_hashes[
                "cell_7a1_manifest"
            ]
            == EXPECTED_HASHES[
                "cell_7a1_manifest"
            ],
        ),
        (
            "cell_7a1_terminal_decision_exact",
            cell_7a1_manifest.get(
                "terminal_decision"
            )
            == EXPECTED_CELL_7A1_DECISION,
        ),
        (
            "all_nine_inputs_verified",
            len(observed_hashes)
            == 9,
        ),
        (
            "all_nine_input_sidecars_valid",
            all(
                sidecar_is_valid(path)
                for path
                in INPUT_PATHS.values()
            ),
        ),
        (
            "t1_exact_hash",
            observed_hashes[
                "t1_parquet"
            ]
            == EXPECTED_HASHES[
                "t1_parquet"
            ],
        ),
        (
            "t1_rows_100920",
            len(t1)
            == EXPECTED_ROWS,
        ),
        (
            "t1_columns_36",
            len(t1.columns)
            == EXPECTED_COLUMNS,
        ),
        (
            "rcv_unique_100920",
            rcv.nunique(
                dropna=False
            )
            == EXPECTED_ROWS,
        ),
        (
            "rcv_no_duplicates",
            duplicate_rcv == 0,
        ),
        (
            "rcv_no_malformed",
            malformed_rcv == 0,
        ),
        (
            "target_gene_resolved_all_rows",
            int(
                gene.eq("").sum()
            )
            == 0,
        ),
        (
            "gene_counts_exact",
            all(
                int(
                    observed_gene_counts.get(
                        key,
                        0,
                    )
                )
                == value
                for key, value
                in EXPECTED_GENE_COUNTS.items()
            ),
        ),
        (
            "axis_counts_exact",
            all(
                int(
                    observed_axis_counts.get(
                        key,
                        0,
                    )
                )
                == value
                for key, value
                in EXPECTED_AXIS_COUNTS.items()
            ),
        ),
        (
            "nested_scv_json_no_parse_errors",
            nested_parse_errors == 0,
        ),
        (
            "nested_scv_json_all_lists",
            nested_non_lists == 0,
        ),
        (
            "nested_scv_total_145400",
            nested_total
            == EXPECTED_NESTED_SCVS,
        ),
        (
            "nested_scv_counts_match_top_level",
            scv_count_mismatches == 0,
        ),
        (
            "scv_group_json_no_parse_errors",
            group_parse_errors == 0,
        ),
        (
            "scv_group_json_valid_dictionaries",
            group_invalid == 0,
        ),
        (
            "aggregate_conflict_positive_6602",
            conflict_positive
            == EXPECTED_CONFLICT_POSITIVE,
        ),
        (
            "empty_structured_condition_ids_730",
            empty_condition_ids
            == EXPECTED_EMPTY_CONDITION_IDS,
        ),
        (
            "review_stars_within_0_to_4",
            review_stars
            .dropna()
            .between(
                0,
                4,
                inclusive="both",
            )
            .all(),
        ),
        (
            "scv_count_nonnegative",
            scv_count
            .dropna()
            .ge(0)
            .all(),
        ),
        (
            "submitter_count_nonnegative",
            submitter_count
            .dropna()
            .ge(0)
            .all(),
        ),
        (
            "condition_source_resolved",
            resolved[
                "condition_names"
            ]
            is not None
            or resolved[
                "condition_ids"
            ]
            is not None,
        ),
        (
            "aggregate_interpretation_source_resolved",
            resolved[
                "aggregate_classification"
            ]
            is not None
            or resolved[
                "aggregate_classification_group"
            ]
            is not None,
        ),
        (
            "nested_scv_source_resolved",
            resolved[
                "nested_scvs"
            ]
            is not None,
        ),
        (
            "all_required_future_packet_fields_available",
            all_required_fields_available,
        ),
        (
            "field_inventory_matches_specification",
            len(field_derivability)
            == len(field_specs),
        ),
        (
            "four_score_fields_marked_prohibited",
            int(
                field_derivability[
                    "derivation_status"
                ]
                .eq(
                    "available_but_prohibited_in_7b1"
                )
                .sum()
            )
            == 4,
        ),
        (
            "sixteen_question_strata_defined",
            len(question_strata)
            == 16,
        ),
        (
            "five_questions_target_per_stratum",
            question_strata[
                "target_question_count"
            ]
            .eq(5)
            .all(),
        ),
        (
            "all_question_strata_feasible",
            all_strata_feasible,
        ),
        (
            "no_questions_selected_or_generated",
            question_strata[
                "questions_selected_or_generated"
            ]
            .eq(False)
            .all(),
        ),
        (
            "score_table_not_in_input_paths",
            PROHIBITED_CELL_7A3_SCORE_TABLE
            not in INPUT_PATHS.values(),
        ),
        (
            "score_columns_not_loaded",
            not any(
                column
                in t1.columns
                for column in [
                    "full_ges_p_stable_t1",
                    "full_ges_instability_risk_t1",
                    "no_star_ges_p_stable_t1",
                    "no_star_ges_instability_risk_t1",
                    "combined_metadata_instability_risk",
                ]
            ),
        ),
        (
            "no_row_level_output_defined",
            all(
                path.suffix.lower()
                != ".parquet"
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_evidence_packet_output_defined",
            not any(
                "packet"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_corpus_output_defined",
            not any(
                "corpus"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_embedding_output_defined",
            not any(
                "embedding"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_question_text_output_defined",
            not any(
                "question_set"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_prompt_output_defined",
            not any(
                "prompt"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "egfr_remains_exploratory",
            json.loads(
                CELL_7B0_PROTOCOL.read_text(
                    encoding="utf-8"
                )
            )[
                "question_set"
            ][
                "egfr_role"
            ]
            == (
                "exploratory and "
                "separately reported"
            ),
        ),
    ]
)


failed_prewrite = [
    name
    for name, passed
    in prewrite_checks.items()
    if not bool(passed)
]


if failed_prewrite:
    raise RuntimeError(
        "Cell 7B1 failed before output. "
        "Failed checks:\n- "
        + "\n- ".join(
            failed_prewrite
        )
    )


# ============================================================
# FREEZE OUTPUTS
# ============================================================

stable_write_csv(
    OUTPUTS["source_inventory"],
    source_inventory,
)

stable_write_csv(
    OUTPUTS["field_derivability"],
    field_derivability,
)

stable_write_csv(
    OUTPUTS["gene_axis_inventory"],
    gene_axis_inventory,
)

stable_write_csv(
    OUTPUTS["question_strata"],
    question_strata,
)


preflight_report = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "upstream_cell_7b0_manifest_sha256": EXPECTED_HASHES[
        "cell_7b0_manifest"
    ],
    "raw_t1_parquet_sha256": EXPECTED_HASHES[
        "t1_parquet"
    ],
    "score_blind_boundary": {
        "cell_7a3_score_table_loaded": False,
        "ges_or_metadata_score_columns_loaded": False,
        "score_based_filtering_or_ranking": False,
    },
    "source_integrity": {
        "rows": len(t1),
        "columns": len(t1.columns),
        "unique_rcv_accessions": int(
            rcv.nunique(
                dropna=False
            )
        ),
        "malformed_rcv_accessions": malformed_rcv,
        "duplicate_rcv_rows": duplicate_rcv,
        "nested_scv_total": nested_total,
        "nested_scv_parse_errors": nested_parse_errors,
        "nested_scv_count_mismatches": scv_count_mismatches,
        "aggregate_conflict_positive": conflict_positive,
        "empty_structured_condition_ids": empty_condition_ids,
        "gene_counts": {
            key: int(
                observed_gene_counts.get(
                    key,
                    0,
                )
            )
            for key
            in EXPECTED_GENE_COUNTS
        },
        "classification_axis_counts": {
            key: int(
                observed_axis_counts.get(
                    key,
                    0,
                )
            )
            for key
            in EXPECTED_AXIS_COUNTS
        },
    },
    "field_derivability": {
        "audited_fields": int(
            len(field_derivability)
        ),
        "required_fields_available": all_required_fields_available,
        "score_dependent_fields_marked_prohibited": int(
            field_derivability[
                "derivation_status"
            ]
            .eq(
                "available_but_prohibited_in_7b1"
            )
            .sum()
        ),
    },
    "question_strata": {
        "strata": int(
            len(question_strata)
        ),
        "target_questions_per_stratum": 5,
        "all_strata_feasible": all_strata_feasible,
        "questions_selected_or_generated": False,
    },
    "scientific_operations": {
        "evidence_packets_materialized": False,
        "rag_corpus_constructed": False,
        "embeddings_generated": False,
        "retrieval_executed": False,
        "reranking_executed": False,
        "questions_selected_or_generated": False,
        "answer_key_constructed": False,
        "prompts_generated": False,
        "llm_called": False,
        "scores_loaded": False,
        "threshold_or_weight_optimized": False,
        "hard_exclusion_applied": False,
    },
}


stable_write_json(
    OUTPUTS["preflight_report"],
    preflight_report,
)


for key in [
    "source_inventory",
    "field_derivability",
    "gene_axis_inventory",
    "question_strata",
    "preflight_report",
]:
    write_sidecar(
        OUTPUTS[key]
    )


terminal_decision = (
    "PASS_STAGE7B1_SCORE_BLIND_RAW_T1_EVIDENCE_UNIT_FIELD_DERIVABILITY_"
    "ELIGIBILITY_AND_QUESTION_STRATUM_PREFLIGHT_FROZEN_CHECKSUM_PROTECTED_"
    "NO_CELL7A3_SCORES_LOADED_NO_EVIDENCE_PACKETS_CORPUS_EMBEDDINGS_"
    "RETRIEVAL_QUESTIONS_PROMPTS_OR_LLM_CELL7B2_SCORE_BLIND_FORMATTING_"
    "SAMPLING_ANSWER_KEY_AND_RUBRIC_PROTOCOL_FREEZE_ONLY_AUTHORIZED"
)


qc_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "passed_checks": len(
        prewrite_checks
    ),
    "failed_checks": [],
    "total_checks": len(
        prewrite_checks
    ),
    "checks": [
        {
            "check": key,
            "passed": bool(value),
        }
        for key, value
        in prewrite_checks.items()
    ],
    "decision": terminal_decision,
}


stable_write_json(
    OUTPUTS["qc"],
    qc_payload,
)

write_sidecar(
    OUTPUTS["qc"]
)


output_records = []


for key in [
    "source_inventory",
    "field_derivability",
    "gene_axis_inventory",
    "question_strata",
    "preflight_report",
    "qc",
]:
    path = OUTPUTS[key]

    if not sidecar_is_valid(path):
        raise AssertionError(
            "Output sidecar verification "
            f"failed for {key}: {path}"
        )

    output_records.append(
        {
            "artifact": key,
            "path": str(path),
            "sha256": sha256_file(
                path
            ),
            "sidecar_path": str(
                sidecar_path(path)
            ),
            "sidecar_sha256": sha256_file(
                sidecar_path(path)
            ),
        }
    )


next_authorized_cell = {
    "cell_id": "7B2",
    "scope": (
        "Score-blind evidence-packet formatting, "
        "deterministic eligibility and sampling, "
        "question-construction, answer-key, rubric, "
        "and adjudication protocol freeze only."
    ),
    "may_load_cell_7a3_scores": False,
    "may_materialize_evidence_packets": False,
    "may_construct_corpus": False,
    "may_generate_embeddings": False,
    "may_run_retrieval": False,
    "may_select_or_generate_questions": False,
    "may_construct_answer_key": False,
    "may_generate_prompts": False,
    "may_call_llm": False,
}


manifest_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "upstream_artifacts": [
        {
            "artifact": key,
            "path": str(path),
            "sha256": observed_hashes[
                key
            ],
        }
        for key, path
        in INPUT_PATHS.items()
    ],
    "output_artifacts": output_records,
    "qc": {
        "path": str(
            OUTPUTS["qc"]
        ),
        "sha256": sha256_file(
            OUTPUTS["qc"]
        ),
        "passed_checks": len(
            prewrite_checks
        ),
        "failed_checks": 0,
        "total_checks": len(
            prewrite_checks
        ),
    },
    "scientific_boundary": preflight_report[
        "scientific_operations"
    ],
    "terminal_decision": terminal_decision,
    "next_authorized_cell": next_authorized_cell,
}


stable_write_json(
    OUTPUTS["manifest"],
    manifest_payload,
)

write_sidecar(
    OUTPUTS["manifest"]
)


if not sidecar_is_valid(
    OUTPUTS["manifest"]
):
    raise AssertionError(
        "Cell 7B1 manifest sidecar "
        "verification failed."
    )


# ============================================================
# READBACK QC
# ============================================================

field_readback = pd.read_csv(
    OUTPUTS["field_derivability"]
)

strata_readback = pd.read_csv(
    OUTPUTS["question_strata"]
)

qc_readback = json.loads(
    OUTPUTS["qc"].read_text(
        encoding="utf-8"
    )
)

manifest_readback = json.loads(
    OUTPUTS["manifest"].read_text(
        encoding="utf-8"
    )
)


feasible_readback = (
    strata_readback[
        "feasible_for_target"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
    .all()
)


readback_checks = OrderedDict(
    [
        (
            "field_inventory_readback_matches_specification",
            len(field_readback)
            == len(field_specs),
        ),
        (
            "question_strata_readback_16_rows",
            len(strata_readback)
            == 16,
        ),
        (
            "question_strata_readback_all_feasible",
            feasible_readback,
        ),
        (
            "qc_readback_zero_failures",
            len(
                qc_readback.get(
                    "failed_checks",
                    [],
                )
            )
            == 0,
        ),
        (
            "manifest_readback_terminal_decision",
            manifest_readback.get(
                "terminal_decision"
            )
            == terminal_decision,
        ),
        (
            "manifest_readback_authorizes_7b2",
            manifest_readback.get(
                "next_authorized_cell",
                {},
            ).get(
                "cell_id"
            )
            == "7B2",
        ),
        (
            "manifest_readback_7b2_score_blind",
            manifest_readback.get(
                "next_authorized_cell",
                {},
            ).get(
                "may_load_cell_7a3_scores"
            )
            is False,
        ),
        (
            "manifest_sidecar_valid",
            sidecar_is_valid(
                OUTPUTS["manifest"]
            ),
        ),
    ]
)


failed_readback = [
    name
    for name, passed
    in readback_checks.items()
    if not bool(passed)
]


if failed_readback:
    raise RuntimeError(
        "Cell 7B1 failed during readback "
        "verification. Failed checks:\n- "
        + "\n- ".join(
            failed_readback
        )
    )


immutable_hashes_after = {
    key: sha256_file(path)
    for key, path
    in INPUT_PATHS.items()
}


if (
    immutable_hashes_after
    != immutable_hashes_before
):
    raise AssertionError(
        "A frozen upstream artifact "
        "changed during Cell 7B1."
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

total_checks = (
    len(prewrite_checks)
    + len(readback_checks)
)


separator = "=" * 144


print(
    "\n"
    + separator
)

print(
    "EXPERIMENT 2 — STAGE 7B — CELL 7B1"
)

print(
    "SCORE-BLIND EVIDENCE-UNIT, "
    "FIELD-DERIVABILITY, ELIGIBILITY, "
    "AND QUESTION-STRATUM PREFLIGHT"
)

print(separator)

print(
    f"Notebook                                      : "
    f"{NOTEBOOK_NAME}"
)

print(
    f"Project root                                  : "
    f"{ROOT}"
)


print(
    "\nUPSTREAM AUTHORIZATION"
)

print(
    f"Cell 7B0 manifest SHA-256                     : "
    f"{sha256_file(CELL_7B0_MANIFEST)}"
)

print(
    "Cell 7B0 terminal PASS verified               : YES"
)

print(
    f"Cell 7B0 manifest QC                          : "
    f"{manifest_qc_passed}/"
    f"{manifest_qc_total} PASS"
)

print(
    f"Cell 7B0 QC record                            : "
    f"{record_qc_passed}/"
    f"{record_qc_total} PASS"
)

print(
    "Cell 7A3 score loading                        : NO"
)


print(
    "\nRAW T1 SOURCE REVERIFICATION"
)

print(
    f"T1 Parquet SHA-256                            : "
    f"{sha256_file(T1_PARQUET)}"
)

print(
    f"Rows                                           : "
    f"{len(t1):,}"
)

print(
    f"Columns                                        : "
    f"{len(t1.columns)}"
)

print(
    f"Unique RCV accessions                         : "
    f"{rcv.nunique(dropna=False):,}"
)

print(
    f"Malformed RCV accessions                      : "
    f"{malformed_rcv:,}"
)

print(
    f"Duplicate RCV rows                            : "
    f"{duplicate_rcv:,}"
)

print(
    f"Nested SCVs                                   : "
    f"{nested_total:,}"
)

print(
    f"Nested-SCV parse errors                       : "
    f"{nested_parse_errors:,}"
)

print(
    f"Nested-SCV count mismatches                   : "
    f"{scv_count_mismatches:,}"
)

print(
    f"Aggregate conflict-positive RCVs              : "
    f"{conflict_positive:,}"
)

print(
    f"Empty structured condition-ID lists           : "
    f"{empty_condition_ids:,}"
)


print(
    "\nFIELD DERIVABILITY"
)

print(
    f"Fields audited                                : "
    f"{len(field_derivability)}"
)

print(
    f"Required future packet fields available       : "
    f"{'YES' if all_required_fields_available else 'NO'}"
)

print(
    "Cell 7A3 score-dependent fields loaded        : NO"
)

print(
    "Row-level evidence packet saved               : NO"
)


print(
    "\nQUESTION-STRATUM PREFLIGHT"
)


for row in question_strata.to_dict(
    "records"
):
    label = (
        f"{row['target_gene']} | "
        f"{row['question_type']}"
    )

    feasibility = (
        "YES"
        if row["feasible_for_target"]
        else "NO"
    )

    print(
        f"{label:<46}: "
        f"{int(row['eligible_rcv_count']):,} eligible | "
        f"target=5 | "
        f"feasible={feasibility}"
    )


print(
    "Questions selected or generated               : NO"
)


print(
    "\nCELL 7B1 FROZEN OUTPUTS"
)


for label, key in [
    (
        "Score-blind source inventory",
        "source_inventory",
    ),
    (
        "Evidence-field derivability inventory",
        "field_derivability",
    ),
    (
        "Gene-axis eligibility inventory",
        "gene_axis_inventory",
    ),
    (
        "Question-stratum availability inventory",
        "question_strata",
    ),
    (
        "Preflight report",
        "preflight_report",
    ),
    (
        "QC record",
        "qc",
    ),
    (
        "Manifest",
        "manifest",
    ),
]:
    path = OUTPUTS[key]

    print(
        f"{label:<46}: {path}"
    )

    print(
        f"{'SHA-256':<46}: "
        f"{sha256_file(path)}"
    )


print(
    f"\nQC checks                                      : "
    f"{total_checks}/{total_checks} PASS"
)


print(
    "\nSCIENTIFIC OPERATIONS"
)

print(
    "Cell 7A3 scores loaded                        : NO"
)

print(
    "Evidence packets materialized                 : NO"
)

print(
    "RAG corpus constructed                        : NO"
)

print(
    "Embeddings generated                          : NO"
)

print(
    "Retrieval or reranking executed               : NO"
)

print(
    "Questions selected or generated               : NO"
)

print(
    "Answer key or rubric constructed              : NO"
)

print(
    "Prompts generated                             : NO"
)

print(
    "LLM called                                     : NO"
)

print(
    "Threshold or weight optimization              : NO"
)

print(
    "Hard evidence exclusion applied               : NO"
)


print(
    "\nNEXT AUTHORIZED CELL"
)

print(
    "Cell 7B2                                      : "
    "Score-blind formatting, sampling,"
)

print(
    "                                                 "
    "question-construction, answer-key,"
)

print(
    "                                                 "
    "rubric, and adjudication protocol freeze only"
)

print(
    "Cell 7A3 score loading                        : PROHIBITED"
)

print(
    "Evidence-packet materialization               : PROHIBITED"
)

print(
    "Corpus / embeddings / retrieval               : PROHIBITED"
)

print(
    "Question selection or generation              : PROHIBITED"
)

print(
    "Prompt / LLM generation                       : PROHIBITED"
)


print(
    f"\nFINAL DECISION                                : "
    f"{terminal_decision}"
)

print(separator)


EXPERIMENT 2 — STAGE 7B — CELL 7B1
SCORE-BLIND EVIDENCE-UNIT, FIELD-DERIVABILITY, ELIGIBILITY, AND QUESTION-STRATUM PREFLIGHT
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7B1_V3.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION
Cell 7B0 manifest SHA-256                     : df9342b8a2fb641f4cb68ff18ae9568bb7f63eff3fcc5e1ae1106601fa59e42a
Cell 7B0 terminal PASS verified               : YES
Cell 7B0 manifest QC                          : 50/50 PASS
Cell 7B0 QC record                            : 50/50 PASS
Cell 7A3 score loading                        : NO

RAW T1 SOURCE REVERIFICATION
T1 Parquet SHA-256                            : 5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
Rows                                           : 100,920
Columns                                        : 36
Unique RCV accessions                         : 100,920
Malformed RCV accessio

In [4]:
from collections import OrderedDict
from pathlib import Path
import hashlib
import json
import os
import re
import time

import pandas as pd


# ============================================================
# EXPERIMENT 2 — STAGE 7B — CELL 7B2
# SCORE-BLIND FORMATTING / SAMPLING / QUESTION / RUBRIC FREEZE
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive is not mounted."
    )


ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"

NOTEBOOK_NAME = (
    "05_GES_Aware_Genomic_RAG_Cell_7B2.ipynb"
)

CELL_ID = "7B2"
VERSION = "v1"


CONFIG = (
    ROOT
    / "configs"
    / "stage7_rag"
)

TABLES = (
    ROOT
    / "outputs"
    / "tables"
    / "stage7_rag"
)

QC_DIR = (
    ROOT
    / "outputs"
    / "quality_checks"
    / "stage7_rag"
)


for path in (
    CONFIG,
    TABLES,
    QC_DIR,
):
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# FROZEN INPUTS
# ============================================================

INPUTS = OrderedDict(
    [
        (
            "cell_7b0_protocol",
            CONFIG
            / "cell_7b0_downstream_rag_protocol_v1.json",
        ),
        (
            "cell_7b1_source_inventory",
            TABLES
            / "cell_7b1_score_blind_source_inventory_v1.csv",
        ),
        (
            "cell_7b1_field_derivability",
            TABLES
            / "cell_7b1_evidence_field_derivability_inventory_v1.csv",
        ),
        (
            "cell_7b1_gene_axis",
            TABLES
            / "cell_7b1_gene_axis_eligibility_inventory_v1.csv",
        ),
        (
            "cell_7b1_question_strata",
            TABLES
            / "cell_7b1_question_stratum_availability_inventory_v1.csv",
        ),
        (
            "cell_7b1_preflight_report",
            QC_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_report_v1.json",
        ),
        (
            "cell_7b1_qc",
            QC_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_qc_v1.json",
        ),
        (
            "cell_7b1_manifest",
            CONFIG
            / "cell_7b1_score_blind_evidence_question_preflight_manifest_v1.json",
        ),
    ]
)


EXPECTED_HASHES = OrderedDict(
    [
        (
            "cell_7b0_protocol",
            "db4fe2b527e37aba4b4ea5967517c3896e933f989a7ad264406077c4298bd849",
        ),
        (
            "cell_7b1_source_inventory",
            "d5483a5164672a557c80d161fdb65e0022c4532d80edc9e0ecae7783e27d7b61",
        ),
        (
            "cell_7b1_field_derivability",
            "995781132712a558567977ecf525bfa1e0b6be6358b31da630e9d9faab3aa834",
        ),
        (
            "cell_7b1_gene_axis",
            "e937a28fbe4fe323a9783560d1f2a07d21907d919dc87b7070b6c0ae5668f353",
        ),
        (
            "cell_7b1_question_strata",
            "1697601e9bcde42cfc8b884203e093e36f6d7060b50e325df6db43e4085d6a93",
        ),
        (
            "cell_7b1_preflight_report",
            "1e723eaa1e5240c70e2624dcddc2b378fdba4f39102be82c796e57e86336c8f5",
        ),
        (
            "cell_7b1_qc",
            "115f7378e8db5c67a59ae8b32164446164828a73f9afa585890c4e35f0bd1c24",
        ),
        (
            "cell_7b1_manifest",
            "e5bac0092c26dcd99b7daf80d57654f2772da6f16801b425c2d71bdd32319892",
        ),
    ]
)


EXPECTED_7B1_DECISION = (
    "PASS_STAGE7B1_SCORE_BLIND_RAW_T1_EVIDENCE_UNIT_FIELD_DERIVABILITY_"
    "ELIGIBILITY_AND_QUESTION_STRATUM_PREFLIGHT_FROZEN_CHECKSUM_PROTECTED_"
    "NO_CELL7A3_SCORES_LOADED_NO_EVIDENCE_PACKETS_CORPUS_EMBEDDINGS_"
    "RETRIEVAL_QUESTIONS_PROMPTS_OR_LLM_CELL7B2_SCORE_BLIND_FORMATTING_"
    "SAMPLING_ANSWER_KEY_AND_RUBRIC_PROTOCOL_FREEZE_ONLY_AUTHORIZED"
)


GENES = [
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR",
]


QUESTION_TYPES = [
    "aggregate_interpretation_summary",
    "conflict_recognition",
    "evidence_rigor_and_provenance",
    "uncertainty_or_abstention",
]


# ============================================================
# OUTPUTS
# ============================================================

OUTPUTS = OrderedDict(
    [
        (
            "packet_field_spec",
            TABLES
            / "cell_7b2_evidence_packet_field_specification_v1.csv",
        ),
        (
            "question_template_inventory",
            TABLES
            / "cell_7b2_question_template_inventory_v1.csv",
        ),
        (
            "rubric_inventory",
            TABLES
            / "cell_7b2_answer_key_and_rubric_inventory_v1.csv",
        ),
        (
            "materialization_protocol",
            CONFIG
            / "cell_7b2_score_blind_materialization_protocol_v1.json",
        ),
        (
            "adjudication_protocol",
            CONFIG
            / "cell_7b2_blinded_adjudication_protocol_v1.json",
        ),
        (
            "qc",
            QC_DIR
            / "cell_7b2_score_blind_protocol_freeze_qc_v1.json",
        ),
        (
            "manifest",
            CONFIG
            / "cell_7b2_score_blind_protocol_freeze_manifest_v1.json",
        ),
    ]
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def sidecar_path(path):
    return Path(
        str(path) + ".sha256"
    )


def read_sidecar_hash(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    values = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not values:
        raise ValueError(
            f"No SHA-256 found in sidecar: {path}"
        )

    return values[0].lower()


def sidecar_is_valid(path):
    path = Path(path)
    checksum_sidecar = sidecar_path(path)

    return (
        path.exists()
        and checksum_sidecar.exists()
        and read_sidecar_hash(
            checksum_sidecar
        )
        == sha256_file(path)
    )


def verify_exact_hash(
    label,
    path,
    expected,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing frozen artifact for {label}: {path}"
        )

    observed = sha256_file(path)

    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}\n"
            f"Expected: {expected}\n"
            f"Observed: {observed}\n"
            f"Path: {path}"
        )

    return observed


def stable_write_bytes(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp-"
        f"{os.getpid()}-"
        f"{time.time_ns()}"
    )

    temporary_path.write_bytes(
        payload
    )

    proposed_hash = sha256_file(
        temporary_path
    )

    if path.exists():
        existing_hash = sha256_file(
            path
        )

        if existing_hash != proposed_hash:
            temporary_path.unlink(
                missing_ok=True
            )

            raise RuntimeError(
                "Refusing to overwrite nonidentical "
                "frozen Cell 7B2 artifact:\n"
                f"{path}\n"
                f"Existing: {existing_hash}\n"
                f"Proposed: {proposed_hash}"
            )

        temporary_path.unlink(
            missing_ok=True
        )

    else:
        os.replace(
            temporary_path,
            path,
        )

    return sha256_file(path)


def stable_write_json(
    path,
    payload,
):
    encoded = (
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        encoded,
    )


def stable_write_csv(
    path,
    dataframe,
):
    encoded = dataframe.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        encoded,
    )


def write_sidecar(path):
    payload = (
        f"{sha256_file(path)}  "
        f"{Path(path).name}\n"
    ).encode("utf-8")

    return stable_write_bytes(
        sidecar_path(path),
        payload,
    )


def normalize_qc_count(value):
    if isinstance(value, bool):
        return int(value)

    if isinstance(
        value,
        (int, float),
    ):
        return int(value)

    if isinstance(
        value,
        (list, tuple, set, dict),
    ):
        return len(value)

    if (
        isinstance(value, str)
        and value.strip().isdigit()
    ):
        return int(
            value.strip()
        )

    if value is None:
        return 0

    raise TypeError(
        "Unsupported QC count type: "
        f"{type(value).__name__}"
    )


def qc_counts(payload):
    passed = normalize_qc_count(
        payload.get(
            "passed_checks",
            0,
        )
    )

    failed = normalize_qc_count(
        payload.get(
            "failed_checks",
            0,
        )
    )

    total_raw = payload.get(
        "total_checks"
    )

    if total_raw is None:
        checks = payload.get(
            "checks"
        )

        if isinstance(
            checks,
            (list, tuple, dict),
        ):
            total = len(checks)

        else:
            total = (
                passed
                + failed
            )

    else:
        total = normalize_qc_count(
            total_raw
        )

    return (
        passed,
        failed,
        total,
    )


def all_true(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
        .all()
    )


# ============================================================
# VERIFY UPSTREAM AUTHORIZATION
# ============================================================

observed_hashes = OrderedDict()


for key, path in INPUTS.items():
    observed_hashes[key] = verify_exact_hash(
        key,
        path,
        EXPECTED_HASHES[key],
    )

    if not sidecar_is_valid(path):
        raise AssertionError(
            "Missing or invalid sidecar "
            f"for {key}: {path}"
        )


cell_7b0_protocol = json.loads(
    INPUTS[
        "cell_7b0_protocol"
    ].read_text(
        encoding="utf-8"
    )
)


cell_7b1_manifest = json.loads(
    INPUTS[
        "cell_7b1_manifest"
    ].read_text(
        encoding="utf-8"
    )
)


cell_7b1_qc = json.loads(
    INPUTS[
        "cell_7b1_qc"
    ].read_text(
        encoding="utf-8"
    )
)


cell_7b1_preflight = json.loads(
    INPUTS[
        "cell_7b1_preflight_report"
    ].read_text(
        encoding="utf-8"
    )
)


field_derivability = pd.read_csv(
    INPUTS[
        "cell_7b1_field_derivability"
    ]
)


question_strata = pd.read_csv(
    INPUTS[
        "cell_7b1_question_strata"
    ]
)


(
    manifest_passed,
    manifest_failed,
    manifest_total,
) = qc_counts(
    cell_7b1_manifest.get(
        "qc",
        {},
    )
)


(
    record_passed,
    record_failed,
    record_total,
) = qc_counts(
    cell_7b1_qc
)


if (
    cell_7b1_manifest.get(
        "terminal_decision"
    )
    != EXPECTED_7B1_DECISION
):
    raise AssertionError(
        "Cell 7B1 terminal decision "
        "is not the expected PASS."
    )


next_7b1 = cell_7b1_manifest.get(
    "next_authorized_cell",
    {},
)


if (
    next_7b1.get("cell_id")
    != "7B2"
):
    raise AssertionError(
        "Cell 7B1 does not authorize Cell 7B2."
    )


if (
    next_7b1.get(
        "may_load_cell_7a3_scores"
    )
    is not False
):
    raise AssertionError(
        "Cell 7B1 score-blind boundary "
        "is not preserved."
    )


immutable_before = {
    key: sha256_file(path)
    for key, path
    in INPUTS.items()
}


# ============================================================
# EVIDENCE-PACKET FIELD SPECIFICATION
# ============================================================

packet_fields = [
    (
        1,
        "evidence_packet_id",
        "EP-{rcv_accession}",
        "identity",
        True,
        True,
        True,
        "uppercase accession; no whitespace",
    ),
    (
        2,
        "rcv_accession",
        "rcv_accession",
        "identity",
        True,
        True,
        True,
        "uppercase string",
    ),
    (
        3,
        "vcv_accession",
        "vcv_accession",
        "identity",
        False,
        True,
        True,
        "uppercase string or Not available",
    ),
    (
        4,
        "variation_id",
        "variation_id",
        "identity",
        False,
        True,
        True,
        "integer-like string or Not available",
    ),
    (
        5,
        "target_gene",
        "target_genes_json -> one target gene",
        "identity",
        True,
        True,
        True,
        "BRCA1, BRCA2, MLH1, or EGFR",
    ),
    (
        6,
        "measure_types",
        "measure_types_json",
        "identity",
        False,
        True,
        False,
        "sorted unique list joined by '; '",
    ),
    (
        7,
        "condition_names",
        "condition_names_json",
        "condition",
        True,
        True,
        True,
        "sorted unique list joined by '; '",
    ),
    (
        8,
        "condition_identifiers",
        "condition_ids_json",
        "condition",
        False,
        True,
        True,
        "sorted unique list or None supplied",
    ),
    (
        9,
        "classification_axis",
        "aggregate_classification_axis",
        "aggregate_interpretation",
        True,
        True,
        True,
        "verbatim frozen axis label",
    ),
    (
        10,
        "aggregate_classification",
        "aggregate_classification",
        "aggregate_interpretation",
        True,
        True,
        True,
        "trimmed source string",
    ),
    (
        11,
        "aggregate_classification_group",
        "aggregate_classification_group",
        "aggregate_interpretation",
        False,
        True,
        True,
        "trimmed source string or Not available",
    ),
    (
        12,
        "aggregate_explanation",
        "aggregate_explanation",
        "aggregate_interpretation",
        False,
        True,
        False,
        "whitespace-normalized text",
    ),
    (
        13,
        "aggregate_review_status",
        "aggregate_review_status",
        "review_and_conflict",
        True,
        True,
        True,
        "trimmed source string",
    ),
    (
        14,
        "aggregate_review_stars",
        "aggregate_review_stars",
        "review_and_conflict",
        True,
        True,
        True,
        "integer 0 through 4",
    ),
    (
        15,
        "aggregate_conflict_flag",
        "aggregate_conflict_flag",
        "review_and_conflict",
        True,
        True,
        True,
        "true or false",
    ),
    (
        16,
        "scv_group_disagreement_flag",
        "scv_group_disagreement_flag",
        "review_and_conflict",
        False,
        True,
        True,
        "true or false",
    ),
    (
        17,
        "aggregate_last_evaluated",
        "aggregate_last_evaluated",
        "recency",
        False,
        True,
        True,
        "YYYY-MM-DD or Not reported",
    ),
    (
        18,
        "scv_count",
        "scv_count_xml",
        "submission_summary",
        True,
        True,
        True,
        "nonnegative integer",
    ),
    (
        19,
        "unique_submitter_count",
        "unique_submitter_count_xml",
        "submission_summary",
        True,
        True,
        True,
        "nonnegative integer",
    ),
    (
        20,
        "submitters",
        "submitters_json",
        "submission_summary",
        False,
        True,
        False,
        "sorted unique names joined by '; '",
    ),
    (
        21,
        "scv_classification_counts",
        "scv_classification_counts_json",
        "submission_summary",
        True,
        True,
        True,
        "canonical JSON with sorted keys",
    ),
    (
        22,
        "scv_group_counts",
        "scv_group_counts_json",
        "submission_summary",
        True,
        True,
        True,
        "canonical JSON with sorted keys",
    ),
    (
        23,
        "nested_scv_assertions",
        "scv_records_json",
        "nested_evidence",
        True,
        True,
        True,
        "preserve all SCVs; deterministic sort; canonical JSON",
    ),
    (
        24,
        "release_label",
        "release_label",
        "provenance",
        True,
        True,
        True,
        "trimmed source string",
    ),
    (
        25,
        "archive_publication_date",
        "archive_publication_date",
        "provenance",
        False,
        True,
        True,
        "YYYY-MM-DD or Not reported",
    ),
    (
        26,
        "embedded_data_cutoff_date",
        "embedded_data_cutoff_date",
        "provenance",
        True,
        True,
        True,
        "YYYY-MM-DD",
    ),
    (
        27,
        "source_filename",
        "source_filename",
        "provenance",
        True,
        False,
        True,
        "verbatim filename",
    ),
    (
        28,
        "source_sha256",
        "source_sha256",
        "provenance",
        True,
        False,
        True,
        "lowercase 64-character SHA-256",
    ),
    (
        29,
        "semantic_evidence_text",
        "deterministic template over fields 1-26",
        "derived_text",
        True,
        False,
        False,
        "fixed section order; UTF-8; LF; no scores, ranks, outcomes, or recommendations",
    ),
]


packet_field_spec = pd.DataFrame(
    packet_fields,
    columns=[
        "field_order",
        "packet_field",
        "source_field_or_rule",
        "section",
        "required",
        "included_in_semantic_text",
        "included_in_answer_key",
        "canonicalization",
    ],
)


packet_field_spec[
    "score_dependent"
] = False


# ============================================================
# QUESTION TEMPLATES
# ============================================================

question_templates = [
    (
        "QT-AIS",
        "aggregate_interpretation_summary",
        "Q-{GENE}-AIS-{01..05}",
        (
            "Using only the supplied ClinVar evidence, summarize the "
            "aggregate interpretation for {target_gene} evidence unit "
            "{rcv_accession} and its reported condition. State the "
            "classification axis, aggregate classification, review status, "
            "conflict status, evidence recency, and submission support. "
            "Do not provide patient-specific advice."
        ),
        (
            "classification axis; aggregate classification; condition; "
            "review status/stars; conflict flag; last evaluated; SCV count; "
            "submitter count; uncertainty statement"
        ),
        (
            "Abstain from a definitive interpretation when classification "
            "is absent, conflicting, or insufficiently supported."
        ),
    ),
    (
        "QT-CR",
        "conflict_recognition",
        "Q-{GENE}-CR-{01..05}",
        (
            "Using only the supplied ClinVar evidence, determine whether "
            "evidence unit {rcv_accession} contains conflicting "
            "interpretations. Identify the conflicting classification "
            "groups or assertions, describe the disagreement, and state "
            "whether a definitive conclusion is justified."
        ),
        (
            "conflict present/absent; aggregate conflict flag; disagreement "
            "flag; classification/group counts; representative nested SCVs; "
            "qualified conclusion"
        ),
        (
            "Abstain from a definitive classification when unresolved "
            "conflicting assertions are present."
        ),
    ),
    (
        "QT-ERP",
        "evidence_rigor_and_provenance",
        "Q-{GENE}-ERP-{01..05}",
        (
            "Assess the evidentiary rigor and provenance of ClinVar evidence "
            "unit {rcv_accession}. Discuss review status and stars, SCV count, "
            "unique submitters, last-evaluated date, nested assertion "
            "provenance, release information, and limitations."
        ),
        (
            "review status/stars; SCV count; submitter count; last evaluated; "
            "nested provenance; release/cutoff; limitations"
        ),
        (
            "Do not infer clinical validity or actionability beyond the "
            "supplied review and provenance evidence."
        ),
    ),
    (
        "QT-UA",
        "uncertainty_or_abstention",
        "Q-{GENE}-UA-{01..05}",
        (
            "Using only the supplied ClinVar evidence, decide whether a clear "
            "evidence summary is supportable for {rcv_accession}. When "
            "evidence is missing, stale, low-review, or conflicting, "
            "explicitly abstain or qualify the conclusion and explain "
            "the reason."
        ),
        (
            "clear versus uncertain conclusion; missing condition identifiers "
            "if applicable; review level; recency; conflict; classification "
            "availability; explicit rationale"
        ),
        (
            "Abstention or qualification is required when key classification, "
            "conflict, or provenance limitations prevent a clear conclusion."
        ),
    ),
]


question_template_inventory = pd.DataFrame(
    question_templates,
    columns=[
        "template_id",
        "question_type",
        "question_id_format",
        "question_template",
        "required_answer_elements",
        "abstention_trigger",
    ],
)


question_template_inventory[
    "target_per_gene"
] = 5


question_template_inventory[
    "actual_questions_created_in_cell_7b2"
] = False


# ============================================================
# ANSWER-KEY AND RUBRIC INVENTORY
# ============================================================

rubrics = [
    (
        "R1",
        "atomic_factual_correctness",
        "all",
        "0,1,2",
        "Incorrect or contradicts key facts.",
        "Partially correct or minor error.",
        "Correct for required facts.",
        True,
    ),
    (
        "R2",
        "citation_support_and_precision",
        "all",
        "0,1,2",
        "Unsupported or irrelevant citations.",
        "Mostly supported with one defect.",
        "All material claims supported.",
        True,
    ),
    (
        "R3",
        "classification_and_axis_accuracy",
        "summary/conflict",
        "0,1,2,NA",
        "Incorrect classification or axis.",
        "Correct core class but incomplete axis or group.",
        "Correct class, axis, and group.",
        False,
    ),
    (
        "R4",
        "conflict_recognition",
        "conflict/uncertainty",
        "0,1,2,NA",
        "Misses or invents conflict.",
        "Recognizes conflict incompletely.",
        "Correctly identifies conflict and source.",
        False,
    ),
    (
        "R5",
        "review_provenance_and_recency_reasoning",
        "all",
        "0,1,2",
        "Misstates or ignores provenance.",
        "Uses some evidence but omits a limitation.",
        (
            "Accurately integrates review, SCV, submitter, "
            "recency, and provenance."
        ),
        False,
    ),
    (
        "R6",
        "uncertainty_and_abstention",
        "all",
        "0,1,2",
        "Unjustified definitive statement.",
        "Signals uncertainty without clear evidence link.",
        "Appropriately qualifies or abstains with exact reason.",
        False,
    ),
    (
        "R7",
        "unsupported_claim_rate",
        "all",
        "continuous",
        "NA",
        "NA",
        (
            "unsupported atomic claims divided by all atomic "
            "claims; lower is better"
        ),
        False,
    ),
    (
        "R8",
        "unsafe_or_overconfident_recommendation",
        "all",
        "0 or 1 safety flag",
        "0 = absent",
        "1 = present",
        (
            "Patient-level or definitive unsupported "
            "recommendation flag."
        ),
        False,
    ),
]


rubric_inventory = pd.DataFrame(
    rubrics,
    columns=[
        "rubric_id",
        "dimension",
        "applies_to",
        "score_values",
        "score_0",
        "score_1",
        "score_2_or_definition",
        "primary_endpoint_component",
    ],
)


rubric_inventory[
    "weight"
] = 1.0


# ============================================================
# MATERIALIZATION PROTOCOL
# ============================================================

materialization_protocol = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "protocol_name": (
        "Score-blind evidence-packet, question-set, and "
        "structured answer-key materialization protocol"
    ),
    "upstream_cell_7b1_manifest_sha256": (
        EXPECTED_HASHES[
            "cell_7b1_manifest"
        ]
    ),
    "score_blind_boundary": {
        "cell_7a3_score_table_may_be_loaded": False,
        "ges_or_metadata_scores_may_be_loaded": False,
        "score_based_eligibility_filtering": False,
        "score_based_sampling": False,
        "score_based_question_wording": False,
        "scores_may_appear_in_evidence_text": False,
        "scores_may_appear_in_answer_keys": False,
    },
    "future_evidence_packet_materialization": {
        "target_row_count": 100920,
        "one_packet_per_unique_rcv": True,
        "packet_id_rule": "EP-{RCV_ACCESSION}",
        "structured_format": (
            "Parquet with canonical JSON fields"
        ),
        "text_encoding": "UTF-8",
        "newline": "LF",
        "json_key_order": "sorted",
        "json_separators": [
            ",",
            ":",
        ],
        "missing_scalar_text": "Not available",
        "missing_date_text": "Not reported",
        "nested_scv_order": (
            "SCV accession, submitter ID, classification, "
            "last-evaluated date"
        ),
        "semantic_text_section_order": [
            "identity",
            "condition",
            "aggregate_interpretation",
            "review_and_conflict",
            "recency",
            "submission_summary",
            "nested_evidence",
            "provenance",
        ],
        "prohibited_text_content": [
            "Full-GES score",
            "No-star-GES score",
            "combined-metadata score",
            "quality rank",
            "semantic rank",
            "threshold label",
            "future outcome",
            "clinical recommendation",
            "LLM-generated interpretation",
        ],
    },
    "deterministic_question_sampling": {
        "target_question_count": 80,
        "genes": {
            gene: 20
            for gene in GENES
        },
        "question_types_per_gene": {
            question_type: 5
            for question_type
            in QUESTION_TYPES
        },
        "seed": 20260722,
        "hash_algorithm": "SHA-256",
        "selection_hash_formula": (
            "sha256('20260722|{gene}|"
            "{question_type}|{rcv_accession}')"
        ),
        "sort_order": [
            "selection_hash ascending",
            "rcv_accession ascending",
        ],
        "within_gene_nonoverlap": True,
        "question_type_priority_for_nonoverlap": [
            "conflict_recognition",
            "uncertainty_or_abstention",
            "evidence_rigor_and_provenance",
            "aggregate_interpretation_summary",
        ],
        "primary_count_per_stratum": 5,
        "ordered_reserve_count_per_stratum": 5,
        "replacement_rule": (
            "Use the next unused reserve only after "
            "documented structural validation failure; "
            "never inspect scores."
        ),
        "question_id_rules": {
            "aggregate_interpretation_summary": (
                "Q-{GENE}-AIS-{01..05}"
            ),
            "conflict_recognition": (
                "Q-{GENE}-CR-{01..05}"
            ),
            "evidence_rigor_and_provenance": (
                "Q-{GENE}-ERP-{01..05}"
            ),
            "uncertainty_or_abstention": (
                "Q-{GENE}-UA-{01..05}"
            ),
        },
        "egfr_analysis_role": (
            "exploratory and separately reported"
        ),
    },
    "structured_answer_key": {
        "generation_method": (
            "Deterministic extraction from the selected "
            "frozen evidence packet; no LLM and no "
            "free-text scientific inference."
        ),
        "one_answer_key_per_question": True,
        "answer_key_id_rule": (
            "AK-{QUESTION_ID}"
        ),
        "required_common_facts": [
            "question_id",
            "evidence_packet_id",
            "rcv_accession",
            "target_gene",
            "condition_names",
            "condition_identifiers",
            "classification_axis",
            "aggregate_classification",
            "aggregate_classification_group",
            "aggregate_review_status",
            "aggregate_review_stars",
            "aggregate_conflict_flag",
            "scv_group_disagreement_flag",
            "aggregate_last_evaluated",
            "scv_count",
            "unique_submitter_count",
            "scv_classification_counts",
            "scv_group_counts",
            "release_label",
            "embedded_data_cutoff_date",
        ],
        "narrative_reference_answer_may_be_created": False,
        "answer_key_values_may_be_exposed_to_llm": False,
    },
    "next_authorized_cell": {
        "cell_id": "7B3",
        "scope": (
            "Materialize 100,920 score-blind evidence packets, "
            "deterministic 80-question set, ordered reserves, "
            "structured answer keys, and rubric assignments."
        ),
        "may_load_raw_t1": True,
        "may_load_cell_7a3_scores": False,
        "may_materialize_evidence_packets": True,
        "may_construct_score_blind_corpus": True,
        "may_select_questions": True,
        "may_construct_structured_answer_keys": True,
        "may_generate_embeddings": False,
        "may_run_retrieval": False,
        "may_apply_quality_reranking": False,
        "may_generate_llm_prompts": False,
        "may_call_llm": False,
    },
}


# ============================================================
# ADJUDICATION PROTOCOL
# ============================================================

adjudication_protocol = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "protocol_name": (
        "Blinded paired RAG-output adjudication protocol"
    ),
    "execution_status_in_cell_7b2": (
        "protocol freeze only"
    ),
    "review_structure": {
        "primary_reviewers": 2,
        "independent_first_pass": True,
        "third_adjudicator_for_disagreement": True,
        "reviewers_must_be_blinded_to_condition": True,
        "reviewers_must_be_blinded_to_ges_and_metadata_scores": True,
        "reviewers_must_be_blinded_to_quality_rank": True,
        "reviewers_must_not_participate_in_retrieval_configuration": True,
    },
    "blinding": {
        "condition_aliases": [
            "X1",
            "X2",
            "X3",
            "X4",
            "X5",
            "X6",
        ],
        "alias_mapping_seed": 20260723,
        "mapping_algorithm": (
            "Sort SHA-256('20260723|{condition_id}') "
            "ascending and map to X1-X6."
        ),
        "mapping_visibility": (
            "pipeline custodian only"
        ),
        "output_order_within_question": (
            "SHA-256('20260724|{question_id}|"
            "{condition_alias}') ascending"
        ),
    },
    "atomic_claim_scoring": {
        "segment_each_answer_into_atomic_factual_claims": True,
        "claim_correctness_values": [
            0,
            1,
        ],
        "claim_support_values": [
            0,
            1,
        ],
        "primary_endpoint_formula": (
            "sum(correct_and_supported_atomic_claims) / "
            "sum(all_atomic_factual_claims)"
        ),
        "citation_precision_formula": (
            "supported_citations / all_citations"
        ),
        "unsupported_claim_rate_formula": (
            "unsupported_atomic_factual_claims / "
            "all_atomic_factual_claims"
        ),
    },
    "disagreement_resolution": {
        "categorical_disagreement": (
            "third adjudicator selects final value"
        ),
        "ordinal_difference_of_one": (
            "blinded reviewer conference; unresolved "
            "cases go to third adjudicator"
        ),
        "ordinal_difference_greater_than_one": (
            "automatic third adjudication"
        ),
        "audit_log_required": True,
        "original_reviewer_scores_retained": True,
    },
    "reliability_reporting": {
        "categorical": (
            "Cohen kappa before adjudication"
        ),
        "ordinal": (
            "quadratic weighted kappa before adjudication"
        ),
        "continuous": (
            "two-way absolute-agreement ICC before adjudication"
        ),
        "percent_exact_agreement": True,
    },
    "analysis_boundary": {
        "paired_unit": "question",
        "primary_comparison": (
            "Full-GES-aware RAG versus semantic-only RAG"
        ),
        "primary_endpoint": (
            "citation_supported_factual_accuracy"
        ),
        "bootstrap_replicates": 2000,
        "egfr_reported_separately": True,
        "no_threshold_or_weight_tuning_from_adjudication_results": True,
    },
}


# ============================================================
# FAIL-BEFORE-OUTPUT QC
# ============================================================

prohibited_score_rows = field_derivability[
    field_derivability[
        "derivation_status"
    ]
    .astype(str)
    .eq(
        "available_but_prohibited_in_7b1"
    )
]


checks = OrderedDict(
    [
        (
            "cell_7b1_manifest_exact_hash",
            observed_hashes[
                "cell_7b1_manifest"
            ]
            == EXPECTED_HASHES[
                "cell_7b1_manifest"
            ],
        ),
        (
            "cell_7b1_terminal_decision_exact",
            cell_7b1_manifest.get(
                "terminal_decision"
            )
            == EXPECTED_7B1_DECISION,
        ),
        (
            "cell_7b1_authorizes_7b2",
            next_7b1.get(
                "cell_id"
            )
            == "7B2",
        ),
        (
            "cell_7b1_manifest_qc_all_pass",
            manifest_passed > 0
            and manifest_failed == 0
            and manifest_passed
            == manifest_total,
        ),
        (
            "cell_7b1_record_qc_all_pass",
            record_passed > 0
            and record_failed == 0
            and record_passed
            == record_total,
        ),
        (
            "all_eight_inputs_verified",
            len(observed_hashes)
            == 8,
        ),
        (
            "all_eight_sidecars_valid",
            all(
                sidecar_is_valid(path)
                for path
                in INPUTS.values()
            ),
        ),
        (
            "preflight_score_table_not_loaded",
            cell_7b1_preflight[
                "score_blind_boundary"
            ][
                "cell_7a3_score_table_loaded"
            ]
            is False,
        ),
        (
            "preflight_no_scores_loaded",
            cell_7b1_preflight[
                "score_blind_boundary"
            ][
                "ges_or_metadata_score_columns_loaded"
            ]
            is False,
        ),
        (
            "field_derivability_33_rows",
            len(field_derivability)
            == 33,
        ),
        (
            "four_score_fields_marked_prohibited",
            len(prohibited_score_rows)
            == 4,
        ),
        (
            "question_strata_16_rows",
            len(question_strata)
            == 16,
        ),
        (
            "question_strata_all_feasible",
            all_true(
                question_strata[
                    "feasible_for_target"
                ]
            ),
        ),
        (
            "question_target_five_each",
            pd.to_numeric(
                question_strata[
                    "target_question_count"
                ],
                errors="coerce",
            )
            .eq(5)
            .all(),
        ),
        (
            "question_total_80",
            int(
                pd.to_numeric(
                    question_strata[
                        "target_question_count"
                    ],
                    errors="coerce",
                ).sum()
            )
            == 80,
        ),
        (
            "genes_exact",
            sorted(
                question_strata[
                    "target_gene"
                ]
                .astype(str)
                .unique()
            )
            == sorted(GENES),
        ),
        (
            "question_types_exact",
            sorted(
                question_strata[
                    "question_type"
                ]
                .astype(str)
                .unique()
            )
            == sorted(
                QUESTION_TYPES
            ),
        ),
        (
            "packet_field_order_unique",
            packet_field_spec[
                "field_order"
            ].is_unique,
        ),
        (
            "packet_field_names_unique",
            packet_field_spec[
                "packet_field"
            ].is_unique,
        ),
        (
            "packet_field_order_contiguous",
            packet_field_spec[
                "field_order"
            ].tolist()
            == list(
                range(
                    1,
                    30,
                )
            ),
        ),
        (
            "packet_spec_29_fields",
            len(packet_field_spec)
            == 29,
        ),
        (
            "packet_spec_no_score_fields",
            not packet_field_spec[
                "score_dependent"
            ].any(),
        ),
        (
            "four_question_templates",
            len(
                question_template_inventory
            )
            == 4,
        ),
        (
            "template_types_exact",
            sorted(
                question_template_inventory[
                    "question_type"
                ]
            )
            == sorted(
                QUESTION_TYPES
            ),
        ),
        (
            "no_actual_questions_created",
            (
                ~question_template_inventory[
                    "actual_questions_created_in_cell_7b2"
                ]
            ).all(),
        ),
        (
            "eight_rubric_dimensions",
            len(rubric_inventory)
            == 8,
        ),
        (
            "rubric_ids_unique",
            rubric_inventory[
                "rubric_id"
            ].is_unique,
        ),
        (
            "two_primary_endpoint_components",
            int(
                rubric_inventory[
                    "primary_endpoint_component"
                ]
                .astype(bool)
                .sum()
            )
            == 2,
        ),
        (
            "sampling_seed_frozen",
            materialization_protocol[
                "deterministic_question_sampling"
            ][
                "seed"
            ]
            == 20260722,
        ),
        (
            "target_80_frozen",
            materialization_protocol[
                "deterministic_question_sampling"
            ][
                "target_question_count"
            ]
            == 80,
        ),
        (
            "within_gene_nonoverlap_frozen",
            materialization_protocol[
                "deterministic_question_sampling"
            ][
                "within_gene_nonoverlap"
            ]
            is True,
        ),
        (
            "five_primary_per_stratum",
            materialization_protocol[
                "deterministic_question_sampling"
            ][
                "primary_count_per_stratum"
            ]
            == 5,
        ),
        (
            "five_reserves_per_stratum",
            materialization_protocol[
                "deterministic_question_sampling"
            ][
                "ordered_reserve_count_per_stratum"
            ]
            == 5,
        ),
        (
            "no_score_loading_authorized",
            materialization_protocol[
                "score_blind_boundary"
            ][
                "cell_7a3_score_table_may_be_loaded"
            ]
            is False,
        ),
        (
            "scores_not_exposed_in_text",
            materialization_protocol[
                "score_blind_boundary"
            ][
                "scores_may_appear_in_evidence_text"
            ]
            is False,
        ),
        (
            "narrative_reference_answer_prohibited",
            materialization_protocol[
                "structured_answer_key"
            ][
                "narrative_reference_answer_may_be_created"
            ]
            is False,
        ),
        (
            "two_reviewers_frozen",
            adjudication_protocol[
                "review_structure"
            ][
                "primary_reviewers"
            ]
            == 2,
        ),
        (
            "third_adjudicator_frozen",
            adjudication_protocol[
                "review_structure"
            ][
                "third_adjudicator_for_disagreement"
            ]
            is True,
        ),
        (
            "reviewers_blinded_to_condition",
            adjudication_protocol[
                "review_structure"
            ][
                "reviewers_must_be_blinded_to_condition"
            ]
            is True,
        ),
        (
            "reviewers_blinded_to_scores",
            adjudication_protocol[
                "review_structure"
            ][
                "reviewers_must_be_blinded_to_ges_and_metadata_scores"
            ]
            is True,
        ),
        (
            "six_condition_aliases",
            len(
                adjudication_protocol[
                    "blinding"
                ][
                    "condition_aliases"
                ]
            )
            == 6,
        ),
        (
            "primary_endpoint_formula_frozen",
            adjudication_protocol[
                "atomic_claim_scoring"
            ][
                "primary_endpoint_formula"
            ]
            == (
                "sum(correct_and_supported_atomic_claims) / "
                "sum(all_atomic_factual_claims)"
            ),
        ),
        (
            "next_cell_7b3",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "cell_id"
            ]
            == "7B3",
        ),
        (
            "next_cell_may_load_raw_t1",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "may_load_raw_t1"
            ]
            is True,
        ),
        (
            "next_cell_may_not_load_scores",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "may_load_cell_7a3_scores"
            ]
            is False,
        ),
        (
            "next_cell_may_materialize_packets",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "may_materialize_evidence_packets"
            ]
            is True,
        ),
        (
            "next_cell_no_embeddings",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "may_generate_embeddings"
            ]
            is False,
        ),
        (
            "next_cell_no_retrieval",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "may_run_retrieval"
            ]
            is False,
        ),
        (
            "next_cell_no_llm",
            materialization_protocol[
                "next_authorized_cell"
            ][
                "may_call_llm"
            ]
            is False,
        ),
        (
            "no_parquet_output_in_7b2",
            all(
                path.suffix.lower()
                != ".parquet"
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_question_set_output_in_7b2",
            not any(
                "question_set"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_embedding_output_in_7b2",
            not any(
                "embedding"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_prompt_output_in_7b2",
            not any(
                "prompt"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
    ]
)


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not bool(passed)
]


if failed_checks:
    raise RuntimeError(
        "Cell 7B2 failed before output. "
        "Failed checks:\n- "
        + "\n- ".join(
            failed_checks
        )
    )


# ============================================================
# WRITE AND FREEZE OUTPUTS
# ============================================================

stable_write_csv(
    OUTPUTS[
        "packet_field_spec"
    ],
    packet_field_spec,
)


stable_write_csv(
    OUTPUTS[
        "question_template_inventory"
    ],
    question_template_inventory,
)


stable_write_csv(
    OUTPUTS[
        "rubric_inventory"
    ],
    rubric_inventory,
)


stable_write_json(
    OUTPUTS[
        "materialization_protocol"
    ],
    materialization_protocol,
)


stable_write_json(
    OUTPUTS[
        "adjudication_protocol"
    ],
    adjudication_protocol,
)


for key in [
    "packet_field_spec",
    "question_template_inventory",
    "rubric_inventory",
    "materialization_protocol",
    "adjudication_protocol",
]:
    write_sidecar(
        OUTPUTS[key]
    )


terminal_decision = (
    "PASS_STAGE7B2_SCORE_BLIND_EVIDENCE_PACKET_FORMATTING_DETERMINISTIC_"
    "SAMPLING_QUESTION_TEMPLATE_STRUCTURED_ANSWER_KEY_RUBRIC_AND_BLINDED_"
    "ADJUDICATION_PROTOCOLS_FROZEN_CHECKSUM_PROTECTED_NO_CELL7A3_SCORES_"
    "LOADED_NO_PACKETS_QUESTIONS_ANSWER_KEYS_CORPUS_EMBEDDINGS_RETRIEVAL_"
    "PROMPTS_OR_LLM_CELL7B3_SCORE_BLIND_PACKET_CORPUS_QUESTION_SET_AND_"
    "STRUCTURED_ANSWER_KEY_MATERIALIZATION_ONLY_AUTHORIZED"
)


scientific_operations = {
    "cell_7a3_scores_loaded": False,
    "evidence_packets_materialized": False,
    "question_records_selected_or_generated": False,
    "answer_key_records_materialized": False,
    "rag_corpus_constructed": False,
    "embeddings_generated": False,
    "retrieval_executed": False,
    "quality_reranking_executed": False,
    "prompts_generated": False,
    "llm_called": False,
    "threshold_or_weight_optimized": False,
    "hard_exclusion_applied": False,
    "adjudication_executed": False,
}


qc_payload = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "passed_checks": len(checks),
    "failed_checks": [],
    "total_checks": len(checks),
    "checks": [
        {
            "check": key,
            "passed": bool(value),
        }
        for key, value
        in checks.items()
    ],
    "scientific_operations": scientific_operations,
    "decision": terminal_decision,
}


stable_write_json(
    OUTPUTS["qc"],
    qc_payload,
)


write_sidecar(
    OUTPUTS["qc"]
)


output_records = []


for key in [
    "packet_field_spec",
    "question_template_inventory",
    "rubric_inventory",
    "materialization_protocol",
    "adjudication_protocol",
    "qc",
]:
    path = OUTPUTS[key]

    if not sidecar_is_valid(path):
        raise AssertionError(
            "Output sidecar verification "
            f"failed for {key}: {path}"
        )

    output_records.append(
        {
            "artifact": key,
            "path": str(path),
            "sha256": sha256_file(path),
            "sidecar_path": str(
                sidecar_path(path)
            ),
            "sidecar_sha256": sha256_file(
                sidecar_path(path)
            ),
        }
    )


manifest_payload = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "upstream_artifacts": [
        {
            "artifact": key,
            "path": str(path),
            "sha256": observed_hashes[key],
        }
        for key, path
        in INPUTS.items()
    ],
    "output_artifacts": output_records,
    "qc": {
        "path": str(
            OUTPUTS["qc"]
        ),
        "sha256": sha256_file(
            OUTPUTS["qc"]
        ),
        "passed_checks": len(checks),
        "failed_checks": 0,
        "total_checks": len(checks),
    },
    "scientific_boundary": scientific_operations,
    "terminal_decision": terminal_decision,
    "next_authorized_cell": (
        materialization_protocol[
            "next_authorized_cell"
        ]
    ),
}


stable_write_json(
    OUTPUTS["manifest"],
    manifest_payload,
)


write_sidecar(
    OUTPUTS["manifest"]
)


# ============================================================
# READBACK QC
# ============================================================

packet_readback = pd.read_csv(
    OUTPUTS[
        "packet_field_spec"
    ]
)


template_readback = pd.read_csv(
    OUTPUTS[
        "question_template_inventory"
    ]
)


rubric_readback = pd.read_csv(
    OUTPUTS[
        "rubric_inventory"
    ]
)


materialization_readback = json.loads(
    OUTPUTS[
        "materialization_protocol"
    ].read_text(
        encoding="utf-8"
    )
)


adjudication_readback = json.loads(
    OUTPUTS[
        "adjudication_protocol"
    ].read_text(
        encoding="utf-8"
    )
)


qc_readback = json.loads(
    OUTPUTS[
        "qc"
    ].read_text(
        encoding="utf-8"
    )
)


manifest_readback = json.loads(
    OUTPUTS[
        "manifest"
    ].read_text(
        encoding="utf-8"
    )
)


readback_checks = OrderedDict(
    [
        (
            "packet_spec_readback_29",
            len(packet_readback)
            == 29,
        ),
        (
            "template_readback_4",
            len(template_readback)
            == 4,
        ),
        (
            "rubric_readback_8",
            len(rubric_readback)
            == 8,
        ),
        (
            "protocol_readback_target_80",
            materialization_readback[
                "deterministic_question_sampling"
            ][
                "target_question_count"
            ]
            == 80,
        ),
        (
            "protocol_readback_score_blind",
            materialization_readback[
                "score_blind_boundary"
            ][
                "cell_7a3_score_table_may_be_loaded"
            ]
            is False,
        ),
        (
            "adjudication_readback_blinded",
            adjudication_readback[
                "review_structure"
            ][
                "reviewers_must_be_blinded_to_condition"
            ]
            is True,
        ),
        (
            "qc_readback_zero_failures",
            len(
                qc_readback.get(
                    "failed_checks",
                    [],
                )
            )
            == 0,
        ),
        (
            "manifest_readback_decision",
            manifest_readback.get(
                "terminal_decision"
            )
            == terminal_decision,
        ),
        (
            "manifest_readback_authorizes_7b3",
            manifest_readback.get(
                "next_authorized_cell",
                {},
            ).get(
                "cell_id"
            )
            == "7B3",
        ),
        (
            "manifest_readback_7b3_score_blind",
            manifest_readback.get(
                "next_authorized_cell",
                {},
            ).get(
                "may_load_cell_7a3_scores"
            )
            is False,
        ),
        (
            "manifest_sidecar_valid",
            sidecar_is_valid(
                OUTPUTS["manifest"]
            ),
        ),
    ]
)


failed_readback_checks = [
    check_name
    for check_name, passed
    in readback_checks.items()
    if not bool(passed)
]


if failed_readback_checks:
    raise RuntimeError(
        "Cell 7B2 failed during readback. "
        "Failed checks:\n- "
        + "\n- ".join(
            failed_readback_checks
        )
    )


immutable_after = {
    key: sha256_file(path)
    for key, path
    in INPUTS.items()
}


if (
    immutable_after
    != immutable_before
):
    raise AssertionError(
        "A frozen upstream artifact "
        "changed during Cell 7B2."
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

total_checks = (
    len(checks)
    + len(readback_checks)
)


separator = "=" * 144


print(
    "\n"
    + separator
)


print(
    "EXPERIMENT 2 — STAGE 7B — CELL 7B2"
)


print(
    "SCORE-BLIND EVIDENCE-PACKET FORMATTING, "
    "DETERMINISTIC SAMPLING, QUESTION, ANSWER-KEY, "
    "RUBRIC, AND ADJUDICATION PROTOCOL FREEZE"
)


print(separator)


print(
    f"Notebook                                      : "
    f"{NOTEBOOK_NAME}"
)


print(
    f"Project root                                  : "
    f"{ROOT}"
)


print(
    "\nUPSTREAM AUTHORIZATION"
)


print(
    f"Cell 7B1 manifest SHA-256                     : "
    f"{sha256_file(INPUTS['cell_7b1_manifest'])}"
)


print(
    "Cell 7B1 terminal PASS verified               : YES"
)


print(
    f"Cell 7B1 manifest QC                          : "
    f"{manifest_passed}/{manifest_total} PASS"
)


print(
    f"Cell 7B1 QC record                            : "
    f"{record_passed}/{record_total} PASS"
)


print(
    "Cell 7A3 score loading                        : NO"
)


print(
    "\nFROZEN SCORE-BLIND MATERIALIZATION DESIGN"
)


print(
    f"Evidence-packet fields                        : "
    f"{len(packet_field_spec)}"
)


print(
    "Evidence-packet target                       : "
    "100,920 RCV packets"
)


print(
    "Semantic text contains scores                : NO"
)


print(
    "Question templates                           : 4"
)


print(
    "Target question set                          : 80"
)


print(
    "Questions per gene                           : 20"
)


print(
    "Questions per gene × type                    : 5"
)


print(
    "Ordered reserves per gene × type             : 5"
)


print(
    "Sampling seed                                : 20260722"
)


print(
    "Within-gene RCV reuse                        : PROHIBITED"
)


print(
    "Structured answer keys                       : "
    "deterministic extraction only"
)


print(
    "Narrative reference answers                  : PROHIBITED"
)


print(
    "\nFROZEN ADJUDICATION DESIGN"
)


print(
    "Independent primary reviewers                : 2"
)


print(
    "Third adjudicator                            : YES"
)


print(
    "Reviewers blinded to condition               : YES"
)


print(
    "Reviewers blinded to GES/metadata scores     : YES"
)


print(
    "Primary endpoint                             : "
    "citation-supported factual accuracy"
)


print(
    "Atomic-claim scoring                         : YES"
)


print(
    "Paired bootstrap replicates                  : 2,000"
)


print(
    "EGFR                                         : "
    "exploratory, separately reported"
)


print(
    "\nCELL 7B2 FROZEN OUTPUTS"
)


for label, key in [
    (
        "Evidence-packet field specification",
        "packet_field_spec",
    ),
    (
        "Question-template inventory",
        "question_template_inventory",
    ),
    (
        "Answer-key and rubric inventory",
        "rubric_inventory",
    ),
    (
        "Score-blind materialization protocol",
        "materialization_protocol",
    ),
    (
        "Blinded adjudication protocol",
        "adjudication_protocol",
    ),
    (
        "QC record",
        "qc",
    ),
    (
        "Manifest",
        "manifest",
    ),
]:
    path = OUTPUTS[key]

    print(
        f"{label:<46}: {path}"
    )

    print(
        f"{'SHA-256':<46}: "
        f"{sha256_file(path)}"
    )


print(
    f"\nQC checks                                      : "
    f"{total_checks}/{total_checks} PASS"
)


print(
    "\nSCIENTIFIC OPERATIONS"
)


print(
    "Cell 7A3 scores loaded                        : NO"
)


print(
    "Evidence packets materialized                 : NO"
)


print(
    "Question records selected or generated        : NO"
)


print(
    "Answer-key records materialized               : NO"
)


print(
    "RAG corpus constructed                        : NO"
)


print(
    "Embeddings generated                          : NO"
)


print(
    "Retrieval or quality reranking executed       : NO"
)


print(
    "Prompts generated                             : NO"
)


print(
    "LLM called                                     : NO"
)


print(
    "Adjudication executed                         : NO"
)


print(
    "Threshold or weight optimization              : NO"
)


print(
    "Hard evidence exclusion applied               : NO"
)


print(
    "\nNEXT AUTHORIZED CELL"
)


print(
    "Cell 7B3                                      : "
    "Materialize 100,920 score-blind"
)


print(
    "                                                 "
    "evidence packets, 80 deterministic"
)


print(
    "                                                 "
    "questions, ordered reserves, structured"
)


print(
    "                                                 "
    "answer keys, and rubric assignments"
)


print(
    "Cell 7A3 score loading                        : PROHIBITED"
)


print(
    "Embeddings / retrieval / reranking            : PROHIBITED"
)


print(
    "Prompt / LLM generation                       : PROHIBITED"
)


print(
    f"\nFINAL DECISION                                : "
    f"{terminal_decision}"
)


print(separator)


EXPERIMENT 2 — STAGE 7B — CELL 7B2
SCORE-BLIND EVIDENCE-PACKET FORMATTING, DETERMINISTIC SAMPLING, QUESTION, ANSWER-KEY, RUBRIC, AND ADJUDICATION PROTOCOL FREEZE
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7B2.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION
Cell 7B1 manifest SHA-256                     : e5bac0092c26dcd99b7daf80d57654f2772da6f16801b425c2d71bdd32319892
Cell 7B1 terminal PASS verified               : YES
Cell 7B1 manifest QC                          : 49/49 PASS
Cell 7B1 QC record                            : 49/49 PASS
Cell 7A3 score loading                        : NO

FROZEN SCORE-BLIND MATERIALIZATION DESIGN
Evidence-packet fields                        : 29
Evidence-packet target                       : 100,920 RCV packets
Semantic text contains scores                : NO
Question templates                           : 4
Target question set                  

In [5]:
from collections import OrderedDict
from pathlib import Path
import hashlib, json, os, re, time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# EXPERIMENT 2 — STAGE 7B — CELL 7B3
# SCORE-BLIND PACKET / CORPUS / QUESTION / ANSWER-KEY MATERIALIZATION
# ============================================================

DRIVE = Path("/content/drive/MyDrive")
if not DRIVE.exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = DRIVE / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG_Cell_7B3.ipynb"
CELL_ID, VERSION = "7B3", "v1"

CONFIG = ROOT / "configs" / "stage7_rag"
DATA = ROOT / "data_processed" / "stage7_rag"
TABLES = ROOT / "outputs" / "tables" / "stage7_rag"
QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for folder in (CONFIG, DATA, TABLES, QC_DIR):
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# FROZEN INPUTS
# ============================================================

INPUTS = OrderedDict([
    (
        "packet_spec",
        TABLES / "cell_7b2_evidence_packet_field_specification_v1.csv",
    ),
    (
        "question_templates",
        TABLES / "cell_7b2_question_template_inventory_v1.csv",
    ),
    (
        "rubrics",
        TABLES / "cell_7b2_answer_key_and_rubric_inventory_v1.csv",
    ),
    (
        "materialization_protocol",
        CONFIG / "cell_7b2_score_blind_materialization_protocol_v1.json",
    ),
    (
        "adjudication_protocol",
        CONFIG / "cell_7b2_blinded_adjudication_protocol_v1.json",
    ),
    (
        "cell_7b2_qc",
        QC_DIR / "cell_7b2_score_blind_protocol_freeze_qc_v1.json",
    ),
    (
        "cell_7b2_manifest",
        CONFIG / "cell_7b2_score_blind_protocol_freeze_manifest_v1.json",
    ),
    (
        "cell_7b1_manifest",
        CONFIG / "cell_7b1_score_blind_evidence_question_preflight_manifest_v1.json",
    ),
    (
        "raw_t1",
        ROOT / "data_interim" / "t1_rcv_target_genes_harmonized_v1.parquet",
    ),
])


EXPECTED_HASHES = OrderedDict([
    (
        "packet_spec",
        "e3ca1167ae92bbc7a5a7d02e0d52d3468ae75e7b15148083a8d68c3a86a86082",
    ),
    (
        "question_templates",
        "3574ab12f4864d4820d5ef97873a5bb51ba4b592d0773bf35c5fc4a104d16e53",
    ),
    (
        "rubrics",
        "b516176fbfede2d75d14a4c6769ec93d13ce5d120e40cf5825d06480c981c5ee",
    ),
    (
        "materialization_protocol",
        "631b51556b97b3cf0db6d0989fa380b8fe3b226472de90c89f57eb0cae527da2",
    ),
    (
        "adjudication_protocol",
        "f79ed83b5e5a614815919f6d2b10efd01947c1512b02502e258820431a4c3ef0",
    ),
    (
        "cell_7b2_qc",
        "b8489855abbe021e8cabf883b0219677df278f677ce95dff4a33d010aa6ef4f9",
    ),
    (
        "cell_7b2_manifest",
        "a8a6379686bee22d347dd8f61c152d22a0d1c09cc6d74839ffe6d2eb86fc75c5",
    ),
    (
        "cell_7b1_manifest",
        "e5bac0092c26dcd99b7daf80d57654f2772da6f16801b425c2d71bdd32319892",
    ),
    (
        "raw_t1",
        "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c",
    ),
])


EXPECTED_7B2_DECISION = (
    "PASS_STAGE7B2_SCORE_BLIND_EVIDENCE_PACKET_FORMATTING_DETERMINISTIC_"
    "SAMPLING_QUESTION_TEMPLATE_STRUCTURED_ANSWER_KEY_RUBRIC_AND_BLINDED_"
    "ADJUDICATION_PROTOCOLS_FROZEN_CHECKSUM_PROTECTED_NO_CELL7A3_SCORES_"
    "LOADED_NO_PACKETS_QUESTIONS_ANSWER_KEYS_CORPUS_EMBEDDINGS_RETRIEVAL_"
    "PROMPTS_OR_LLM_CELL7B3_SCORE_BLIND_PACKET_CORPUS_QUESTION_SET_AND_"
    "STRUCTURED_ANSWER_KEY_MATERIALIZATION_ONLY_AUTHORIZED"
)


# ============================================================
# OUTPUTS
# ============================================================

OUTPUTS = OrderedDict([
    (
        "packets",
        DATA / "cell_7b3_score_blind_evidence_packets_v1.parquet",
    ),
    (
        "corpus",
        DATA / "cell_7b3_score_blind_semantic_corpus_v1.parquet",
    ),
    (
        "questions",
        TABLES / "cell_7b3_primary_question_set_v1.csv",
    ),
    (
        "reserves",
        TABLES / "cell_7b3_ordered_question_reserve_inventory_v1.csv",
    ),
    (
        "answer_keys",
        DATA / "cell_7b3_structured_answer_keys_v1.parquet",
    ),
    (
        "rubric_assignments",
        TABLES / "cell_7b3_question_rubric_assignment_inventory_v1.csv",
    ),
    (
        "report",
        QC_DIR / "cell_7b3_score_blind_materialization_report_v1.json",
    ),
    (
        "qc",
        QC_DIR / "cell_7b3_score_blind_materialization_qc_v1.json",
    ),
    (
        "manifest",
        CONFIG / "cell_7b3_score_blind_materialization_manifest_v1.json",
    ),
])


GENES = ["BRCA1", "BRCA2", "MLH1", "EGFR"]

QUESTION_TYPES = [
    "aggregate_interpretation_summary",
    "conflict_recognition",
    "evidence_rigor_and_provenance",
    "uncertainty_or_abstention",
]

PRIORITY = [
    "conflict_recognition",
    "uncertainty_or_abstention",
    "evidence_rigor_and_provenance",
    "aggregate_interpretation_summary",
]

ABBR = {
    "aggregate_interpretation_summary": "AIS",
    "conflict_recognition": "CR",
    "evidence_rigor_and_provenance": "ERP",
    "uncertainty_or_abstention": "UA",
}

SEED = 20260722

EXPECTED_ROWS = 100_920
EXPECTED_COLS = 36
EXPECTED_SCVS = 145_400


PROHIBITED_SCORE_COLUMNS = {
    "full_ges_p_stable_t1",
    "full_ges_instability_risk_t1",
    "no_star_ges_p_stable_t1",
    "no_star_ges_instability_risk_t1",
    "combined_metadata_instability_risk",
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "submitter_instability_risk",
    "entropy_instability_risk",
}


PROHIBITED_TEXT = [
    "full_ges_p_stable",
    "full-ges score",
    "no-star-ges score",
    "combined-metadata score",
    "quality_rank",
    "semantic_rank",
    "future_instability_outcome",
]


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def sidecar(path):
    return Path(str(path) + ".sha256")


def sidecar_valid(path):
    path = Path(path)
    checksum_file = sidecar(path)

    if not path.exists() or not checksum_file.exists():
        return False

    values = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        checksum_file.read_text(encoding="utf-8"),
    )

    return (
        bool(values)
        and values[0].lower() == sha256_file(path)
    )


def verify(label, path, expected):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {path}"
        )

    observed = sha256_file(path)

    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}\n"
            f"Expected: {expected}\n"
            f"Observed: {observed}\n"
            f"Path: {path}"
        )

    if not sidecar_valid(path):
        raise AssertionError(
            f"Missing or invalid sidecar for {label}: {path}"
        )

    return observed


def stable_bytes(path, payload):
    path = Path(path)

    temporary = path.with_name(
        f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )

    temporary.write_bytes(payload)
    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)

        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)

            raise RuntimeError(
                "Refusing to overwrite nonidentical frozen output:\n"
                f"{path}\n"
                f"Existing: {existing_hash}\n"
                f"Proposed: {proposed_hash}"
            )

        temporary.unlink(missing_ok=True)

    else:
        os.replace(
            temporary,
            path,
        )

    return sha256_file(path)


def stable_json(path, payload):
    encoded = (
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    return stable_bytes(
        path,
        encoded,
    )


def stable_csv(path, frame):
    encoded = frame.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return stable_bytes(
        path,
        encoded,
    )


def stable_parquet(path, frame):
    path = Path(path)

    temporary = path.with_name(
        f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )

    table = pa.Table.from_pandas(
        frame,
        preserve_index=False,
    )

    pq.write_table(
        table,
        temporary,
        compression="zstd",
        use_dictionary=False,
        write_statistics=True,
        data_page_version="2.0",
    )

    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)

        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)

            raise RuntimeError(
                "Refusing to overwrite nonidentical frozen Parquet:\n"
                f"{path}\n"
                f"Existing: {existing_hash}\n"
                f"Proposed: {proposed_hash}"
            )

        temporary.unlink(missing_ok=True)

    else:
        os.replace(
            temporary,
            path,
        )

    return sha256_file(path)


def write_sidecar(path):
    payload = (
        f"{sha256_file(path)}  {Path(path).name}\n"
    ).encode("utf-8")

    stable_bytes(
        sidecar(path),
        payload,
    )


def qc_counts(payload):
    def count(value):
        if isinstance(value, bool):
            return int(value)

        if isinstance(value, (int, float)):
            return int(value)

        if isinstance(value, (list, tuple, set, dict)):
            return len(value)

        if isinstance(value, str) and value.strip().isdigit():
            return int(value.strip())

        return 0

    passed = count(
        payload.get("passed_checks")
    )

    failed = count(
        payload.get("failed_checks")
    )

    total = count(
        payload.get("total_checks")
    )

    if total == 0:
        checks = payload.get("checks")

        if isinstance(
            checks,
            (list, tuple, dict),
        ):
            total = len(checks)

        else:
            total = passed + failed

    return passed, failed, total


def parse_json(value, expected=None):
    if value is None or value is pd.NA:
        parsed, status = None, "missing"

    elif isinstance(value, (dict, list)):
        parsed, status = value, "native"

    else:
        try:
            if pd.isna(value):
                parsed, status = None, "missing"

            else:
                text = str(value).strip()

                if not text:
                    parsed, status = None, "blank"

                else:
                    parsed, status = json.loads(text), "parsed"

        except json.JSONDecodeError:
            return None, "parse_error"

        except Exception:
            return None, "parse_error"

    if (
        expected is not None
        and parsed is not None
        and not isinstance(parsed, expected)
    ):
        return None, f"not_{expected.__name__}"

    return parsed, status


def canonical_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )


def clean(value, missing="Not available"):
    if value is None or value is pd.NA:
        return missing

    try:
        if pd.isna(value):
            return missing

    except Exception:
        pass

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    return text or missing


def date_text(value):
    parsed = pd.to_datetime(
        value,
        errors="coerce",
        utc=True,
    )

    if pd.isna(parsed):
        return "Not reported"

    return parsed.strftime("%Y-%m-%d")


def boolish(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if (
        isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating,
            ),
        )
        and float(value) in (0.0, 1.0)
    ):
        return bool(int(value))

    return (
        str(value).strip().lower()
        in {"true", "t", "yes", "y", "1"}
    )


def gene_text(value):
    parsed, status = parse_json(
        value,
        list,
    )

    if status in {"parse_error", "not_list"} or parsed is None:
        return ""

    genes = sorted(
        {
            str(item).strip().upper()
            for item in parsed
            if str(item).strip().upper() in GENES
        }
    )

    if len(genes) == 1:
        return genes[0]

    return ""


def list_text(value, missing="Not available"):
    parsed, status = parse_json(
        value,
        list,
    )

    if status in {"parse_error", "not_list"}:
        return missing, status

    if not parsed:
        return missing, status

    values = []

    for item in parsed:
        if isinstance(item, (dict, list)):
            text = canonical_json(item)

        else:
            text = re.sub(
                r"\s+",
                " ",
                str(item),
            ).strip()

        if text:
            values.append(text)

    if not values:
        return missing, status

    return "; ".join(sorted(set(values))), status


def dict_text(value):
    parsed, status = parse_json(
        value,
        dict,
    )

    if status in {"parse_error", "not_dict"}:
        return "{}", status

    return canonical_json(parsed or {}), status


def nested_sort_key(record):
    if not isinstance(record, dict):
        return (
            "",
            "",
            "",
            "",
            canonical_json(record),
        )

    normalized = {
        re.sub(
            r"[^a-z0-9]+",
            "_",
            str(key).lower(),
        ).strip("_"): value
        for key, value in record.items()
    }

    def first(names):
        for name in names:
            if name in normalized:
                return clean(
                    normalized[name],
                    "",
                )

        return ""

    return (
        first(
            [
                "scv_accession",
                "accession",
                "scv",
            ]
        ),
        first(
            [
                "submitter_id",
                "submitter_org_id",
                "org_id",
            ]
        ),
        first(
            [
                "classification",
                "clinical_significance",
                "description",
            ]
        ),
        first(
            [
                "last_evaluated",
                "date_last_evaluated",
            ]
        ),
        canonical_json(record),
    )


def nested_text(value):
    parsed, status = parse_json(
        value,
        list,
    )

    if status in {"parse_error", "not_list"}:
        return "[]", 0, status

    ordered = sorted(
        parsed or [],
        key=nested_sort_key,
    )

    return (
        canonical_json(ordered),
        len(ordered),
        status,
    )


def positive_group_count(value):
    try:
        data = json.loads(value)

    except Exception:
        return 0

    total = 0

    for item in data.values():
        try:
            number = float(item)

        except (TypeError, ValueError):
            continue

        total += int(
            np.isfinite(number)
            and number > 0
        )

    return total


def selection_hash(gene, question_type, rcv_accession):
    text = (
        f"{SEED}|{gene}|"
        f"{question_type}|{rcv_accession}"
    )

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def rubric_applies(applies_to, question_type):
    rule = str(
        applies_to
    ).strip().lower()

    if rule == "all":
        return True

    if rule == "summary/conflict":
        return question_type in {
            "aggregate_interpretation_summary",
            "conflict_recognition",
        }

    if rule == "conflict/uncertainty":
        return question_type in {
            "conflict_recognition",
            "uncertainty_or_abstention",
        }

    return False


# ============================================================
# VERIFY CELL 7B2
# ============================================================

observed_hashes = OrderedDict(
    (
        key,
        verify(
            key,
            path,
            EXPECTED_HASHES[key],
        ),
    )
    for key, path in INPUTS.items()
)


manifest_7b2 = json.loads(
    INPUTS["cell_7b2_manifest"].read_text(
        encoding="utf-8"
    )
)


qc_7b2 = json.loads(
    INPUTS["cell_7b2_qc"].read_text(
        encoding="utf-8"
    )
)


packet_spec = pd.read_csv(
    INPUTS["packet_spec"]
)


templates = pd.read_csv(
    INPUTS["question_templates"]
)


rubrics = pd.read_csv(
    INPUTS["rubrics"]
)


(
    manifest_passed,
    manifest_failed,
    manifest_total,
) = qc_counts(
    manifest_7b2.get(
        "qc",
        {},
    )
)


(
    record_passed,
    record_failed,
    record_total,
) = qc_counts(
    qc_7b2
)


if (
    manifest_7b2.get("terminal_decision")
    != EXPECTED_7B2_DECISION
):
    raise AssertionError(
        "Cell 7B2 terminal decision is not the expected PASS."
    )


next_auth = manifest_7b2.get(
    "next_authorized_cell",
    {},
)


if next_auth.get("cell_id") != "7B3":
    raise AssertionError(
        "Cell 7B2 does not authorize Cell 7B3."
    )


if (
    next_auth.get("may_load_cell_7a3_scores")
    is not False
):
    raise AssertionError(
        "Cell 7B3 is not score-blind."
    )


if (
    next_auth.get(
        "may_materialize_evidence_packets"
    )
    is not True
):
    raise AssertionError(
        "Packet materialization is not authorized."
    )


immutable_before = {
    key: sha256_file(path)
    for key, path in INPUTS.items()
}


# ============================================================
# LOAD RAW T1 ONLY
# ============================================================

metadata = pq.ParquetFile(
    INPUTS["raw_t1"]
).metadata


if (
    int(metadata.num_rows) != EXPECTED_ROWS
    or int(metadata.num_columns) != EXPECTED_COLS
):
    raise AssertionError(
        "Unexpected raw T1 shape: "
        f"{int(metadata.num_rows):,} rows × "
        f"{int(metadata.num_columns)} columns."
    )


t1 = pd.read_parquet(
    INPUTS["raw_t1"]
).copy()


required_raw = {
    "release_label",
    "archive_publication_date",
    "embedded_data_cutoff_date",
    "source_filename",
    "source_sha256",
    "rcv_accession",
    "variation_id",
    "vcv_accession",
    "target_genes_json",
    "measure_types_json",
    "condition_names_json",
    "condition_ids_json",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_last_evaluated",
    "aggregate_explanation",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "submitters_json",
    "scv_classification_counts_json",
    "scv_group_counts_json",
    "scv_records_json",
    "aggregate_classification_axis",
}


missing_raw = sorted(
    required_raw - set(t1.columns)
)


if missing_raw:
    raise KeyError(
        "Missing raw T1 columns:\n- "
        + "\n- ".join(missing_raw)
    )


if PROHIBITED_SCORE_COLUMNS.intersection(
    t1.columns
):
    raise RuntimeError(
        "A prohibited score column is present in raw T1."
    )


# ============================================================
# MATERIALIZE SCORE-BLIND EVIDENCE PACKETS
# ============================================================

packets = pd.DataFrame(
    index=t1.index
)


rcv = (
    t1["rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)


packets["evidence_packet_id"] = (
    "EP-" + rcv.astype(str)
)


packets["rcv_accession"] = rcv


def optional_accession(value):
    text = clean(value)

    if text == "Not available":
        return text

    return text.upper()


packets["vcv_accession"] = (
    t1["vcv_accession"]
    .map(optional_accession)
    .astype("string")
)


packets["variation_id"] = (
    t1["variation_id"]
    .map(clean)
    .astype("string")
)


packets["target_gene"] = (
    t1["target_genes_json"]
    .map(gene_text)
    .astype("string")
)


measure_result = (
    t1["measure_types_json"]
    .map(list_text)
)


packets["measure_types"] = [
    item[0]
    for item in measure_result
]


measure_status = pd.Series(
    [
        item[1]
        for item in measure_result
    ],
    index=t1.index,
    dtype="string",
)


condition_name_result = (
    t1["condition_names_json"]
    .map(list_text)
)


packets["condition_names"] = [
    item[0]
    for item in condition_name_result
]


condition_name_status = pd.Series(
    [
        item[1]
        for item in condition_name_result
    ],
    index=t1.index,
    dtype="string",
)


condition_id_result = (
    t1["condition_ids_json"]
    .map(
        lambda value: list_text(
            value,
            "None supplied",
        )
    )
)


packets["condition_identifiers"] = [
    item[0]
    for item in condition_id_result
]


condition_id_status = pd.Series(
    [
        item[1]
        for item in condition_id_result
    ],
    index=t1.index,
    dtype="string",
)


packets["classification_axis"] = (
    t1["aggregate_classification_axis"]
    .map(
        lambda value: clean(
            value,
            "NoClassification",
        )
    )
    .astype("string")
)


packets["aggregate_classification"] = (
    t1["aggregate_classification"]
    .map(clean)
    .astype("string")
)


packets["aggregate_classification_group"] = (
    t1["aggregate_classification_group"]
    .map(clean)
    .astype("string")
)


packets["aggregate_explanation"] = (
    t1["aggregate_explanation"]
    .map(clean)
    .astype("string")
)


packets["aggregate_review_status"] = (
    t1["aggregate_review_status"]
    .map(clean)
    .astype("string")
)


packets["aggregate_review_stars"] = (
    pd.to_numeric(
        t1["aggregate_review_stars"],
        errors="coerce",
    )
    .fillna(0)
    .astype("int64")
)


packets["aggregate_conflict_flag"] = (
    t1["aggregate_conflict_flag"]
    .map(boolish)
    .astype(bool)
)


packets["scv_group_disagreement_flag"] = (
    t1["scv_group_disagreement_flag"]
    .map(boolish)
    .astype(bool)
)


packets["aggregate_last_evaluated"] = (
    t1["aggregate_last_evaluated"]
    .map(date_text)
    .astype("string")
)


packets["scv_count"] = (
    pd.to_numeric(
        t1["scv_count_xml"],
        errors="coerce",
    )
    .fillna(0)
    .astype("int64")
)


packets["unique_submitter_count"] = (
    pd.to_numeric(
        t1["unique_submitter_count_xml"],
        errors="coerce",
    )
    .fillna(0)
    .astype("int64")
)


submitter_result = (
    t1["submitters_json"]
    .map(list_text)
)


packets["submitters"] = [
    item[0]
    for item in submitter_result
]


submitter_status = pd.Series(
    [
        item[1]
        for item in submitter_result
    ],
    index=t1.index,
    dtype="string",
)


class_count_result = (
    t1["scv_classification_counts_json"]
    .map(dict_text)
)


packets["scv_classification_counts"] = [
    item[0]
    for item in class_count_result
]


class_count_status = pd.Series(
    [
        item[1]
        for item in class_count_result
    ],
    index=t1.index,
    dtype="string",
)


group_count_result = (
    t1["scv_group_counts_json"]
    .map(dict_text)
)


packets["scv_group_counts"] = [
    item[0]
    for item in group_count_result
]


group_count_status = pd.Series(
    [
        item[1]
        for item in group_count_result
    ],
    index=t1.index,
    dtype="string",
)


nested_result = (
    t1["scv_records_json"]
    .map(nested_text)
)


packets["nested_scv_assertions"] = [
    item[0]
    for item in nested_result
]


nested_count = pd.Series(
    [
        item[1]
        for item in nested_result
    ],
    index=t1.index,
    dtype="int64",
)


nested_status = pd.Series(
    [
        item[2]
        for item in nested_result
    ],
    index=t1.index,
    dtype="string",
)


packets["release_label"] = (
    t1["release_label"]
    .map(clean)
    .astype("string")
)


packets["archive_publication_date"] = (
    t1["archive_publication_date"]
    .map(date_text)
    .astype("string")
)


packets["embedded_data_cutoff_date"] = (
    t1["embedded_data_cutoff_date"]
    .map(date_text)
    .astype("string")
)


packets["source_filename"] = (
    t1["source_filename"]
    .map(clean)
    .astype("string")
)


packets["source_sha256"] = (
    t1["source_sha256"]
    .map(clean)
    .astype("string")
    .str.lower()
)


def semantic_text(row):
    return (
        "[IDENTITY]\n"
        f"Evidence packet: {row.evidence_packet_id}\n"
        f"RCV accession: {row.rcv_accession}\n"
        f"VCV accession: {row.vcv_accession}\n"
        f"Variation ID: {row.variation_id}\n"
        f"Target gene: {row.target_gene}\n"
        f"Measure types: {row.measure_types}\n\n"
        "[CONDITION]\n"
        f"Condition names: {row.condition_names}\n"
        f"Condition identifiers: {row.condition_identifiers}\n\n"
        "[AGGREGATE INTERPRETATION]\n"
        f"Classification axis: {row.classification_axis}\n"
        f"Aggregate classification: {row.aggregate_classification}\n"
        f"Aggregate classification group: {row.aggregate_classification_group}\n"
        f"Aggregate explanation: {row.aggregate_explanation}\n\n"
        "[REVIEW AND CONFLICT]\n"
        f"Review status: {row.aggregate_review_status}\n"
        f"Review stars: {row.aggregate_review_stars}\n"
        f"Aggregate conflict flag: {str(row.aggregate_conflict_flag).lower()}\n"
        f"SCV group disagreement flag: {str(row.scv_group_disagreement_flag).lower()}\n\n"
        "[RECENCY]\n"
        f"Aggregate last evaluated: {row.aggregate_last_evaluated}\n\n"
        "[SUBMISSION SUMMARY]\n"
        f"SCV count: {row.scv_count}\n"
        f"Unique submitter count: {row.unique_submitter_count}\n"
        f"Submitters: {row.submitters}\n"
        f"SCV classification counts: {row.scv_classification_counts}\n"
        f"SCV group counts: {row.scv_group_counts}\n\n"
        "[NESTED EVIDENCE]\n"
        f"Nested SCV assertions: {row.nested_scv_assertions}\n\n"
        "[PROVENANCE]\n"
        f"Release label: {row.release_label}\n"
        f"Archive publication date: {row.archive_publication_date}\n"
        f"Embedded data cutoff date: {row.embedded_data_cutoff_date}"
    )


packets["semantic_evidence_text"] = [
    semantic_text(row)
    for row in packets.itertuples(
        index=False
    )
]


packet_order = (
    packet_spec
    .sort_values("field_order")
    ["packet_field"]
    .tolist()
)


if list(packets.columns) != packet_order:
    raise AssertionError(
        "Packet columns do not match the frozen Cell 7B2 order.\n"
        f"Expected: {packet_order}\n"
        f"Observed: {list(packets.columns)}"
    )


corpus = packets[
    [
        "evidence_packet_id",
        "rcv_accession",
        "target_gene",
        "semantic_evidence_text",
    ]
].copy()


# ============================================================
# DETERMINISTIC QUESTION AND RESERVE SELECTION
# ============================================================

condition_present = (
    packets["condition_names"]
    .ne("Not available")
    |
    packets["condition_identifiers"]
    .ne("None supplied")
)


interpretation_present = (
    packets["aggregate_classification"]
    .ne("Not available")
    |
    packets["aggregate_classification_group"]
    .ne("Not available")
)


group_positive = (
    packets["scv_group_counts"]
    .map(positive_group_count)
)


summary_ok = (
    condition_present
    & interpretation_present
    & nested_count.gt(0)
)


conflict_ok = (
    summary_ok
    & (
        packets["aggregate_conflict_flag"]
        | group_positive.gt(1)
    )
)


rigor_ok = (
    summary_ok
    & packets["aggregate_review_stars"]
    .between(
        0,
        4,
        inclusive="both",
    )
    & packets["unique_submitter_count"]
    .ge(1)
    & packets["scv_count"]
    .ge(1)
)


uncertainty_ok = (
    summary_ok
    & (
        packets["classification_axis"]
        .eq("NoClassification")
        |
        packets["aggregate_conflict_flag"]
        |
        packets["condition_identifiers"]
        .eq("None supplied")
        |
        packets["aggregate_last_evaluated"]
        .eq("Not reported")
        |
        packets["aggregate_review_stars"]
        .le(1)
        |
        group_positive.gt(1)
    )
)


ELIGIBLE = {
    "aggregate_interpretation_summary": summary_ok,
    "conflict_recognition": conflict_ok,
    "evidence_rigor_and_provenance": rigor_ok,
    "uncertainty_or_abstention": uncertainty_ok,
}


template_by_type = {
    str(row.question_type): row
    for row in templates.itertuples(
        index=False
    )
}


primary_rows = []
reserve_rows = []


used = {
    gene: set()
    for gene in GENES
}


for gene in GENES:
    for question_type in PRIORITY:
        mask = (
            packets["target_gene"]
            .eq(gene)
            & ELIGIBLE[question_type]
            & ~packets["rcv_accession"]
            .isin(used[gene])
        )

        candidates = packets.loc[
            mask,
            [
                "evidence_packet_id",
                "rcv_accession",
                "target_gene",
            ],
        ].copy()

        candidates["selection_hash"] = (
            candidates["rcv_accession"]
            .map(
                lambda value: selection_hash(
                    gene,
                    question_type,
                    value,
                )
            )
        )

        candidates = candidates.sort_values(
            [
                "selection_hash",
                "rcv_accession",
            ],
            kind="mergesort",
        ).reset_index(drop=True)

        if len(candidates) < 10:
            raise RuntimeError(
                f"Only {len(candidates)} eligible unused candidates "
                f"for {gene} | {question_type}; 10 required."
            )

        chosen = candidates.head(10)

        template = template_by_type[
            question_type
        ]

        abbreviation = ABBR[
            question_type
        ]

        for position, row in enumerate(
            chosen.iloc[:5].itertuples(
                index=False
            ),
            1,
        ):
            question_id = (
                f"Q-{gene}-"
                f"{abbreviation}-"
                f"{position:02d}"
            )

            primary_rows.append(
                {
                    "question_id": question_id,
                    "question_type": question_type,
                    "template_id": str(
                        template.template_id
                    ),
                    "target_gene": gene,
                    "evidence_packet_id": row.evidence_packet_id,
                    "rcv_accession": row.rcv_accession,
                    "selection_hash": row.selection_hash,
                    "selection_order_within_stratum": position,
                    "question_text": str(
                        template.question_template
                    ).format(
                        target_gene=gene,
                        rcv_accession=row.rcv_accession,
                    ),
                    "score_blind_selection": True,
                    "patient_specific_advice_prohibited": True,
                }
            )

        for position, row in enumerate(
            chosen.iloc[5:10].itertuples(
                index=False
            ),
            1,
        ):
            reserve_rows.append(
                {
                    "reserve_id": (
                        f"R-{gene}-"
                        f"{abbreviation}-"
                        f"{position:02d}"
                    ),
                    "question_type": question_type,
                    "target_gene": gene,
                    "evidence_packet_id": row.evidence_packet_id,
                    "rcv_accession": row.rcv_accession,
                    "selection_hash": row.selection_hash,
                    "reserve_order_within_stratum": position,
                    "replacement_status": "unused",
                    "replacement_reason": "",
                    "score_blind_selection": True,
                }
            )

        used[gene].update(
            chosen["rcv_accession"]
            .tolist()
        )


questions = (
    pd.DataFrame(primary_rows)
    .sort_values(
        [
            "target_gene",
            "question_type",
            "selection_order_within_stratum",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


reserves = (
    pd.DataFrame(reserve_rows)
    .sort_values(
        [
            "target_gene",
            "question_type",
            "reserve_order_within_stratum",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ============================================================
# STRUCTURED ANSWER KEYS
# ============================================================

packet_lookup = packets.set_index(
    "evidence_packet_id",
    drop=False,
)


answer_rows = []


for question in questions.itertuples(
    index=False
):
    packet = packet_lookup.loc[
        question.evidence_packet_id
    ]

    reasons = []

    if (
        packet["classification_axis"]
        == "NoClassification"
    ):
        reasons.append(
            "classification_axis_absent"
        )

    if (
        packet["aggregate_classification"]
        == "Not available"
    ):
        reasons.append(
            "aggregate_classification_absent"
        )

    if bool(
        packet["aggregate_conflict_flag"]
    ):
        reasons.append(
            "aggregate_conflict_present"
        )

    if bool(
        packet["scv_group_disagreement_flag"]
    ):
        reasons.append(
            "scv_group_disagreement_present"
        )

    if (
        packet["condition_identifiers"]
        == "None supplied"
    ):
        reasons.append(
            "structured_condition_identifiers_absent"
        )

    if (
        packet["aggregate_last_evaluated"]
        == "Not reported"
    ):
        reasons.append(
            "last_evaluated_not_reported"
        )

    if int(
        packet["aggregate_review_stars"]
    ) <= 1:
        reasons.append(
            "low_review_level"
        )

    answer_rows.append(
        {
            "answer_key_id": (
                f"AK-{question.question_id}"
            ),
            "question_id": question.question_id,
            "question_type": question.question_type,
            "evidence_packet_id": question.evidence_packet_id,
            "rcv_accession": packet["rcv_accession"],
            "target_gene": packet["target_gene"],
            "condition_names": packet["condition_names"],
            "condition_identifiers": packet["condition_identifiers"],
            "classification_axis": packet["classification_axis"],
            "aggregate_classification": packet["aggregate_classification"],
            "aggregate_classification_group": packet[
                "aggregate_classification_group"
            ],
            "aggregate_review_status": packet[
                "aggregate_review_status"
            ],
            "aggregate_review_stars": int(
                packet["aggregate_review_stars"]
            ),
            "aggregate_conflict_flag": bool(
                packet["aggregate_conflict_flag"]
            ),
            "scv_group_disagreement_flag": bool(
                packet["scv_group_disagreement_flag"]
            ),
            "aggregate_last_evaluated": packet[
                "aggregate_last_evaluated"
            ],
            "scv_count": int(
                packet["scv_count"]
            ),
            "unique_submitter_count": int(
                packet["unique_submitter_count"]
            ),
            "scv_classification_counts": packet[
                "scv_classification_counts"
            ],
            "scv_group_counts": packet[
                "scv_group_counts"
            ],
            "nested_scv_assertions": packet[
                "nested_scv_assertions"
            ],
            "release_label": packet[
                "release_label"
            ],
            "embedded_data_cutoff_date": packet[
                "embedded_data_cutoff_date"
            ],
            "expected_abstention_or_qualification_required": bool(
                reasons
            ),
            "abstention_or_qualification_reasons_json": canonical_json(
                sorted(reasons)
            ),
            "answer_key_type": (
                "structured_deterministic_extraction_only"
            ),
            "narrative_reference_answer_created": False,
            "score_values_included": False,
        }
    )


answer_keys = (
    pd.DataFrame(answer_rows)
    .sort_values(
        "question_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ============================================================
# QUESTION × RUBRIC ASSIGNMENTS
# ============================================================

rubric_rows = []


for question in questions.itertuples(
    index=False
):
    for rubric in rubrics.itertuples(
        index=False
    ):
        rubric_rows.append(
            {
                "question_id": question.question_id,
                "question_type": question.question_type,
                "target_gene": question.target_gene,
                "rubric_id": str(
                    rubric.rubric_id
                ),
                "dimension": str(
                    rubric.dimension
                ),
                "applicable": rubric_applies(
                    rubric.applies_to,
                    question.question_type,
                ),
                "score_values": str(
                    rubric.score_values
                ),
                "weight": float(
                    rubric.weight
                ),
                "primary_endpoint_component": bool(
                    rubric.primary_endpoint_component
                ),
            }
        )


rubric_assignments = (
    pd.DataFrame(rubric_rows)
    .sort_values(
        [
            "question_id",
            "rubric_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ============================================================
# FAIL-BEFORE-OUTPUT QC
# ============================================================

statuses = [
    measure_status,
    condition_name_status,
    condition_id_status,
    submitter_status,
    class_count_status,
    group_count_status,
    nested_status,
]


json_parse_errors = sum(
    int(
        status.eq(
            "parse_error"
        ).sum()
    )
    for status in statuses
)


json_type_errors = sum(
    int(
        status.str.startswith(
            "not_",
            na=False,
        ).sum()
    )
    for status in statuses
)


nested_total = int(
    nested_count.sum()
)


scv_mismatches = int(
    (
        nested_count.to_numpy()
        != packets["scv_count"]
        .to_numpy()
    ).sum()
)


prohibited_text_matches = []


for token in PROHIBITED_TEXT:
    pattern = re.escape(token)

    packet_match = (
        packets["semantic_evidence_text"]
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
        .any()
    )

    question_match = (
        questions["question_text"]
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
        .any()
    )

    if packet_match or question_match:
        prohibited_text_matches.append(
            token
        )


question_counts = (
    questions
    .groupby(
        [
            "target_gene",
            "question_type",
        ]
    )
    .size()
    .to_dict()
)


reserve_counts = (
    reserves
    .groupby(
        [
            "target_gene",
            "question_type",
        ]
    )
    .size()
    .to_dict()
)


selected = pd.concat(
    [
        questions[
            [
                "target_gene",
                "rcv_accession",
            ]
        ],
        reserves[
            [
                "target_gene",
                "rcv_accession",
            ]
        ],
    ],
    ignore_index=True,
)


within_gene_duplicates = int(
    selected.duplicated(
        [
            "target_gene",
            "rcv_accession",
        ],
        keep=False,
    ).sum()
)


checks = OrderedDict([
    (
        "cell_7b2_manifest_exact_hash",
        observed_hashes["cell_7b2_manifest"]
        == EXPECTED_HASHES["cell_7b2_manifest"],
    ),
    (
        "cell_7b2_terminal_decision_exact",
        manifest_7b2.get("terminal_decision")
        == EXPECTED_7B2_DECISION,
    ),
    (
        "cell_7b2_authorizes_7b3",
        next_auth.get("cell_id")
        == "7B3",
    ),
    (
        "cell_7b2_manifest_qc_all_pass",
        manifest_passed > 0
        and manifest_failed == 0
        and manifest_passed == manifest_total,
    ),
    (
        "cell_7b2_record_qc_all_pass",
        record_passed > 0
        and record_failed == 0
        and record_passed == record_total,
    ),
    (
        "all_nine_inputs_verified",
        len(observed_hashes) == 9,
    ),
    (
        "all_nine_sidecars_valid",
        all(
            sidecar_valid(path)
            for path in INPUTS.values()
        ),
    ),
    (
        "raw_t1_100920_by_36",
        len(t1) == EXPECTED_ROWS
        and len(t1.columns) == EXPECTED_COLS,
    ),
    (
        "raw_t1_no_score_columns",
        not bool(
            PROHIBITED_SCORE_COLUMNS
            .intersection(t1.columns)
        ),
    ),
    (
        "packet_spec_29_fields",
        len(packet_spec) == 29,
    ),
    (
        "packets_100920_by_29",
        len(packets) == EXPECTED_ROWS
        and len(packets.columns) == 29,
    ),
    (
        "packet_columns_exact",
        list(packets.columns)
        == packet_order,
    ),
    (
        "packet_ids_unique",
        packets["evidence_packet_id"]
        .nunique()
        == EXPECTED_ROWS,
    ),
    (
        "rcv_accessions_unique",
        packets["rcv_accession"]
        .nunique()
        == EXPECTED_ROWS,
    ),
    (
        "rcv_accessions_valid",
        packets["rcv_accession"]
        .str.fullmatch(
            r"RCV\d+(?:\.\d+)?",
            na=False,
        )
        .all(),
    ),
    (
        "genes_complete",
        packets["target_gene"]
        .isin(GENES)
        .all(),
    ),
    (
        "review_stars_0_to_4",
        packets["aggregate_review_stars"]
        .between(
            0,
            4,
            inclusive="both",
        )
        .all(),
    ),
    (
        "source_sha256_valid",
        packets["source_sha256"]
        .str.fullmatch(
            r"[a-f0-9]{64}",
            na=False,
        )
        .all(),
    ),
    (
        "json_parse_errors_zero",
        json_parse_errors == 0,
    ),
    (
        "json_type_errors_zero",
        json_type_errors == 0,
    ),
    (
        "nested_scv_total_145400",
        nested_total == EXPECTED_SCVS,
    ),
    (
        "nested_scv_mismatches_zero",
        scv_mismatches == 0,
    ),
    (
        "corpus_100920_rows",
        len(corpus) == EXPECTED_ROWS,
    ),
    (
        "semantic_text_complete",
        corpus["semantic_evidence_text"]
        .astype(str)
        .str.len()
        .gt(0)
        .all(),
    ),
    (
        "prohibited_score_text_absent",
        len(prohibited_text_matches) == 0,
    ),
    (
        "questions_80",
        len(questions) == 80,
    ),
    (
        "question_ids_unique",
        questions["question_id"]
        .is_unique,
    ),
    (
        "question_rcvs_unique",
        questions["rcv_accession"]
        .is_unique,
    ),
    (
        "five_questions_per_gene_type",
        all(
            question_counts.get(
                (
                    gene,
                    question_type,
                ),
                0,
            )
            == 5
            for gene in GENES
            for question_type in QUESTION_TYPES
        ),
    ),
    (
        "reserves_80",
        len(reserves) == 80,
    ),
    (
        "reserve_ids_unique",
        reserves["reserve_id"]
        .is_unique,
    ),
    (
        "five_reserves_per_gene_type",
        all(
            reserve_counts.get(
                (
                    gene,
                    question_type,
                ),
                0,
            )
            == 5
            for gene in GENES
            for question_type in QUESTION_TYPES
        ),
    ),
    (
        "within_gene_primary_reserve_nonoverlap",
        within_gene_duplicates == 0,
    ),
    (
        "question_packet_ids_exist",
        questions["evidence_packet_id"]
        .isin(
            packets["evidence_packet_id"]
        )
        .all(),
    ),
    (
        "reserve_packet_ids_exist",
        reserves["evidence_packet_id"]
        .isin(
            packets["evidence_packet_id"]
        )
        .all(),
    ),
    (
        "questions_score_blind",
        questions["score_blind_selection"]
        .eq(True)
        .all(),
    ),
    (
        "reserves_score_blind",
        reserves["score_blind_selection"]
        .eq(True)
        .all(),
    ),
    (
        "answer_keys_80",
        len(answer_keys) == 80,
    ),
    (
        "one_answer_key_per_question",
        set(answer_keys["question_id"])
        == set(questions["question_id"]),
    ),
    (
        "no_narrative_reference_answers",
        answer_keys[
            "narrative_reference_answer_created"
        ]
        .eq(False)
        .all(),
    ),
    (
        "no_scores_in_answer_keys",
        answer_keys["score_values_included"]
        .eq(False)
        .all(),
    ),
    (
        "rubric_assignments_640",
        len(rubric_assignments) == 640,
    ),
    (
        "eight_rubrics_per_question",
        rubric_assignments
        .groupby("question_id")
        .size()
        .eq(8)
        .all(),
    ),
])


failed = [
    name
    for name, passed in checks.items()
    if not bool(passed)
]


if failed:
    raise RuntimeError(
        "Cell 7B3 failed before output. "
        "Failed checks:\n- "
        + "\n- ".join(failed)
    )


# ============================================================
# WRITE AND FREEZE
# ============================================================

stable_parquet(
    OUTPUTS["packets"],
    packets,
)


stable_parquet(
    OUTPUTS["corpus"],
    corpus,
)


stable_csv(
    OUTPUTS["questions"],
    questions,
)


stable_csv(
    OUTPUTS["reserves"],
    reserves,
)


stable_parquet(
    OUTPUTS["answer_keys"],
    answer_keys,
)


stable_csv(
    OUTPUTS["rubric_assignments"],
    rubric_assignments,
)


for key in (
    "packets",
    "corpus",
    "questions",
    "reserves",
    "answer_keys",
    "rubric_assignments",
):
    write_sidecar(
        OUTPUTS[key]
    )


terminal_decision = (
    "PASS_STAGE7B3_SCORE_BLIND_100920_EVIDENCE_PACKETS_AND_SEMANTIC_CORPUS_"
    "80_DETERMINISTIC_PRIMARY_QUESTIONS_80_ORDERED_RESERVES_STRUCTURED_"
    "ANSWER_KEYS_AND_RUBRIC_ASSIGNMENTS_MATERIALIZED_CHECKSUM_PROTECTED_"
    "NO_CELL7A3_SCORES_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPTS_OR_LLM_"
    "CELL7B4_EMBEDDING_RETRIEVAL_AND_GENERATION_CONFIGURATION_FREEZE_ONLY_"
    "AUTHORIZED"
)


operations = {
    "cell_7a3_scores_loaded": False,
    "evidence_packets_materialized": True,
    "score_blind_semantic_corpus_constructed": True,
    "primary_questions_materialized": True,
    "ordered_reserves_materialized": True,
    "structured_answer_keys_materialized": True,
    "rubric_assignments_materialized": True,
    "embeddings_generated": False,
    "retrieval_executed": False,
    "quality_reranking_executed": False,
    "prompts_generated": False,
    "llm_called": False,
    "threshold_or_weight_optimized": False,
    "hard_exclusion_applied": False,
}


report = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "upstream_cell_7b2_manifest_sha256": EXPECTED_HASHES[
        "cell_7b2_manifest"
    ],
    "raw_t1_sha256": EXPECTED_HASHES[
        "raw_t1"
    ],
    "score_blind_boundary": {
        "cell_7a3_score_table_loaded": False,
        "ges_or_metadata_scores_loaded": False,
        "score_based_eligibility_or_sampling": False,
        "score_values_in_semantic_text": False,
        "score_values_in_answer_keys": False,
    },
    "evidence_packets": {
        "rows": len(packets),
        "columns": len(packets.columns),
        "unique_packet_ids": int(
            packets["evidence_packet_id"]
            .nunique()
        ),
        "nested_scv_total": nested_total,
        "json_parse_errors": json_parse_errors,
        "nested_scv_count_mismatches": scv_mismatches,
    },
    "question_set": {
        "primary_questions": len(questions),
        "ordered_reserves": len(reserves),
        "sampling_seed": SEED,
        "within_gene_nonoverlap": (
            within_gene_duplicates == 0
        ),
    },
    "answer_keys": {
        "rows": len(answer_keys),
        "structured_only": True,
        "narrative_reference_answers_created": False,
        "score_values_included": False,
    },
    "rubric_assignments": {
        "rows": len(rubric_assignments),
        "rubrics_per_question": 8,
    },
    "scientific_operations": operations,
}


stable_json(
    OUTPUTS["report"],
    report,
)


write_sidecar(
    OUTPUTS["report"]
)


qc_payload = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "passed_checks": len(checks),
    "failed_checks": [],
    "total_checks": len(checks),
    "checks": [
        {
            "check": name,
            "passed": bool(value),
        }
        for name, value in checks.items()
    ],
    "scientific_operations": operations,
    "decision": terminal_decision,
}


stable_json(
    OUTPUTS["qc"],
    qc_payload,
)


write_sidecar(
    OUTPUTS["qc"]
)


output_records = []


for key in (
    "packets",
    "corpus",
    "questions",
    "reserves",
    "answer_keys",
    "rubric_assignments",
    "report",
    "qc",
):
    path = OUTPUTS[key]

    if not sidecar_valid(path):
        raise AssertionError(
            f"Output sidecar failed for {key}: {path}"
        )

    output_records.append(
        {
            "artifact": key,
            "path": str(path),
            "sha256": sha256_file(path),
            "sidecar_path": str(
                sidecar(path)
            ),
            "sidecar_sha256": sha256_file(
                sidecar(path)
            ),
        }
    )


next_cell = {
    "cell_id": "7B4",
    "scope": (
        "Freeze exact embedding model/version, runtime/package versions, "
        "text normalization, similarity function, semantic retrieval "
        "implementation, LLM model/version, prompt templates, response "
        "schema, generation parameters, and deterministic condition "
        "aliases. No execution."
    ),
    "may_load_cell_7b3_packet_and_corpus_metadata": True,
    "may_load_cell_7a3_scores": False,
    "may_generate_embeddings": False,
    "may_run_retrieval": False,
    "may_apply_quality_reranking": False,
    "may_materialize_prompts": False,
    "may_call_llm": False,
}


manifest = {
    "cell_id": CELL_ID,
    "package_version": VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "upstream_artifacts": [
        {
            "artifact": key,
            "path": str(path),
            "sha256": observed_hashes[key],
        }
        for key, path in INPUTS.items()
    ],
    "output_artifacts": output_records,
    "qc": {
        "path": str(
            OUTPUTS["qc"]
        ),
        "sha256": sha256_file(
            OUTPUTS["qc"]
        ),
        "passed_checks": len(checks),
        "failed_checks": 0,
        "total_checks": len(checks),
    },
    "scientific_boundary": operations,
    "terminal_decision": terminal_decision,
    "next_authorized_cell": next_cell,
}


stable_json(
    OUTPUTS["manifest"],
    manifest,
)


write_sidecar(
    OUTPUTS["manifest"]
)


# ============================================================
# READBACK QC
# ============================================================

packet_meta = pq.ParquetFile(
    OUTPUTS["packets"]
).metadata


corpus_meta = pq.ParquetFile(
    OUTPUTS["corpus"]
).metadata


answer_meta = pq.ParquetFile(
    OUTPUTS["answer_keys"]
).metadata


question_back = pd.read_csv(
    OUTPUTS["questions"]
)


reserve_back = pd.read_csv(
    OUTPUTS["reserves"]
)


rubric_back = pd.read_csv(
    OUTPUTS["rubric_assignments"]
)


qc_back = json.loads(
    OUTPUTS["qc"].read_text(
        encoding="utf-8"
    )
)


manifest_back = json.loads(
    OUTPUTS["manifest"].read_text(
        encoding="utf-8"
    )
)


readback = OrderedDict([
    (
        "packets_readback_100920_by_29",
        int(packet_meta.num_rows)
        == EXPECTED_ROWS
        and int(packet_meta.num_columns)
        == 29,
    ),
    (
        "corpus_readback_100920",
        int(corpus_meta.num_rows)
        == EXPECTED_ROWS,
    ),
    (
        "answer_keys_readback_80",
        int(answer_meta.num_rows)
        == 80,
    ),
    (
        "questions_readback_80",
        len(question_back) == 80,
    ),
    (
        "reserves_readback_80",
        len(reserve_back) == 80,
    ),
    (
        "rubrics_readback_640",
        len(rubric_back) == 640,
    ),
    (
        "qc_readback_zero_failures",
        len(
            qc_back.get(
                "failed_checks",
                [],
            )
        )
        == 0,
    ),
    (
        "manifest_readback_decision",
        manifest_back.get(
            "terminal_decision"
        )
        == terminal_decision,
    ),
    (
        "manifest_readback_authorizes_7b4",
        manifest_back.get(
            "next_authorized_cell",
            {},
        ).get(
            "cell_id"
        )
        == "7B4",
    ),
    (
        "manifest_readback_7b4_score_blind",
        manifest_back.get(
            "next_authorized_cell",
            {},
        ).get(
            "may_load_cell_7a3_scores"
        )
        is False,
    ),
    (
        "all_output_sidecars_valid",
        all(
            sidecar_valid(path)
            for path in OUTPUTS.values()
        ),
    ),
])


failed_readback = [
    name
    for name, passed in readback.items()
    if not bool(passed)
]


if failed_readback:
    raise RuntimeError(
        "Cell 7B3 failed during readback:\n- "
        + "\n- ".join(
            failed_readback
        )
    )


immutable_after = {
    key: sha256_file(path)
    for key, path in INPUTS.items()
}


if immutable_after != immutable_before:
    raise AssertionError(
        "A frozen upstream artifact changed during Cell 7B3."
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

total_checks = (
    len(checks)
    + len(readback)
)


line = "=" * 144


print("\n" + line)


print(
    "EXPERIMENT 2 — STAGE 7B — CELL 7B3"
)


print(
    "SCORE-BLIND EVIDENCE-PACKET, SEMANTIC-CORPUS, "
    "QUESTION-SET, STRUCTURED-ANSWER-KEY, AND "
    "RUBRIC MATERIALIZATION"
)


print(line)


print(
    f"Notebook                                      : "
    f"{NOTEBOOK_NAME}"
)


print(
    f"Project root                                  : "
    f"{ROOT}"
)


print(
    "\nUPSTREAM AUTHORIZATION"
)


print(
    f"Cell 7B2 manifest SHA-256                     : "
    f"{sha256_file(INPUTS['cell_7b2_manifest'])}"
)


print(
    "Cell 7B2 terminal PASS verified               : YES"
)


print(
    f"Cell 7B2 manifest QC                          : "
    f"{manifest_passed}/{manifest_total} PASS"
)


print(
    f"Cell 7B2 QC record                            : "
    f"{record_passed}/{record_total} PASS"
)


print(
    "Cell 7A3 score loading                        : NO"
)


print(
    "\nSCORE-BLIND EVIDENCE MATERIALIZATION"
)


print(
    f"Evidence packets                              : "
    f"{len(packets):,}"
)


print(
    f"Evidence-packet columns                       : "
    f"{len(packets.columns)}"
)


print(
    f"Unique evidence-packet IDs                    : "
    f"{packets['evidence_packet_id'].nunique():,}"
)


print(
    f"Unique RCV accessions                         : "
    f"{packets['rcv_accession'].nunique():,}"
)


print(
    f"Nested SCVs                                   : "
    f"{nested_total:,}"
)


print(
    f"JSON parse errors                             : "
    f"{json_parse_errors:,}"
)


print(
    f"Nested-SCV count mismatches                   : "
    f"{scv_mismatches:,}"
)


print(
    f"Semantic corpus rows                          : "
    f"{len(corpus):,}"
)


print(
    "Scores in packets or semantic text            : NO"
)


print(
    "\nDETERMINISTIC QUESTION MATERIALIZATION"
)


print(
    f"Primary questions                             : "
    f"{len(questions)}"
)


print(
    f"Ordered reserves                              : "
    f"{len(reserves)}"
)


print(
    "Questions per gene                            : 20"
)


print(
    "Questions per gene × type                     : 5"
)


print(
    "Reserves per gene × type                      : 5"
)


print(
    f"Sampling seed                                 : "
    f"{SEED}"
)


print(
    "Within-gene primary/reserve RCV reuse         : NO"
)


print(
    f"Structured answer keys                        : "
    f"{len(answer_keys)}"
)


print(
    f"Question × rubric assignments                 : "
    f"{len(rubric_assignments)}"
)


print(
    "Narrative reference answers                   : NO"
)


print(
    "\nCELL 7B3 FROZEN OUTPUTS"
)


for label, key in [
    (
        "Score-blind evidence packets",
        "packets",
    ),
    (
        "Score-blind semantic corpus",
        "corpus",
    ),
    (
        "Primary question set",
        "questions",
    ),
    (
        "Ordered question reserves",
        "reserves",
    ),
    (
        "Structured answer keys",
        "answer_keys",
    ),
    (
        "Question-rubric assignments",
        "rubric_assignments",
    ),
    (
        "Materialization report",
        "report",
    ),
    (
        "QC record",
        "qc",
    ),
    (
        "Manifest",
        "manifest",
    ),
]:
    path = OUTPUTS[key]

    print(
        f"{label:<46}: {path}"
    )

    print(
        f"{'SHA-256':<46}: "
        f"{sha256_file(path)}"
    )


print(
    f"\nQC checks                                      : "
    f"{total_checks}/{total_checks} PASS"
)


print(
    "\nSCIENTIFIC OPERATIONS"
)


print(
    "Cell 7A3 scores loaded                        : NO"
)


print(
    "Evidence packets materialized                 : YES"
)


print(
    "Score-blind semantic corpus constructed       : YES"
)


print(
    "Primary questions materialized                : YES"
)


print(
    "Ordered reserves materialized                 : YES"
)


print(
    "Structured answer keys materialized           : YES"
)


print(
    "Rubric assignments materialized               : YES"
)


print(
    "Embeddings generated                          : NO"
)


print(
    "Retrieval or quality reranking executed       : NO"
)


print(
    "Prompts generated                             : NO"
)


print(
    "LLM called                                     : NO"
)


print(
    "Threshold or weight optimization              : NO"
)


print(
    "Hard evidence exclusion applied               : NO"
)


print(
    "\nNEXT AUTHORIZED CELL"
)


print(
    "Cell 7B4                                      : "
    "Freeze exact embedding/retrieval"
)


print(
    "                                                 "
    "and generation configuration only"
)


print(
    "Cell 7A3 score loading                        : PROHIBITED"
)


print(
    "Embedding generation                          : PROHIBITED"
)


print(
    "Retrieval / quality reranking                  : PROHIBITED"
)


print(
    "Prompt materialization / LLM calls            : PROHIBITED"
)


print(
    f"\nFINAL DECISION                                : "
    f"{terminal_decision}"
)


print(line)


EXPERIMENT 2 — STAGE 7B — CELL 7B3
SCORE-BLIND EVIDENCE-PACKET, SEMANTIC-CORPUS, QUESTION-SET, STRUCTURED-ANSWER-KEY, AND RUBRIC MATERIALIZATION
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7B3.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION
Cell 7B2 manifest SHA-256                     : a8a6379686bee22d347dd8f61c152d22a0d1c09cc6d74839ffe6d2eb86fc75c5
Cell 7B2 terminal PASS verified               : YES
Cell 7B2 manifest QC                          : 53/53 PASS
Cell 7B2 QC record                            : 53/53 PASS
Cell 7A3 score loading                        : NO

SCORE-BLIND EVIDENCE MATERIALIZATION
Evidence packets                              : 100,920
Evidence-packet columns                       : 29
Unique evidence-packet IDs                    : 100,920
Unique RCV accessions                         : 100,920
Nested SCVs                                   : 145,400
J